In [ ]:
#-- Load libraries
library(readr)
library(dplyr)
library(data.table)
library(ggplot2)
library(lubridate)
library(purrr)
library(stringr)
library(tidyverse)
library(openxlsx)
library(nnet)
library(forcats)

In [ ]:
#-- Load outcomes
outcome <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Main_CYP_Outcomes_90days_Phenoconversion_noAD.csv") %>%
  mutate(
    FirstDispense.treatment = as.Date(FirstDispense.treatment),
    ExpectedEndDate.treatment = as.Date(ExpectedEndDate.treatment),
    FirstDispense.treatment.episode = as.Date(FirstDispense.treatment.episode),
    ExpectedEndDate.treatment.episode = as.Date(ExpectedEndDate.treatment.episode),
    FirstDispense.other.episode = as.Date(FirstDispense.other.episode),
    ExpectedEndDate.other.episode = as.Date(ExpectedEndDate.other.episode),
    Incidence = as.Date(Incidence),
    DateFirstDispense = as.Date(DateFirstDispense),
    Index = as.Date(Index)
  )

In [ ]:
#-- First switch (first incidence of switch)
first_switch <- outcome %>%
  filter(FinalClassification.treatment == "Switching") %>%
  group_by(person_id) %>%
  filter(Incidence == min(Incidence)) %>%
  filter(FirstDispense.treatment.episode == min(FirstDispense.treatment.episode)) %>%
  filter(ExpectedEndDate.treatment.episode == max(ExpectedEndDate.treatment.episode)) %>%
  filter(ExpectedEndDate.treatment == min(ExpectedEndDate.treatment)) %>%
  ungroup()

#-- No switch (first incidence of continuation among non-switchers)
no_switch <- outcome %>%
  filter(!person_id %in% first_switch$person_id) %>%
  #filter(FinalClassification.treatment == "Continuation") %>%
  group_by(person_id) %>%
  arrange(Incidence, FirstDispense.treatment) %>%
  slice(1) %>%
  ungroup()

#-- Input data for switching vs. non-switching
dat <- rbind(first_switch, no_switch)


In [ ]:
#-- Read in predicted ancestries
workspace_bucket <- Sys.getenv("WORKSPACE_BUCKET")
anc_file_path <- file.path(workspace_bucket, "ancestry/ancestry_preds.tsv")
anc_df <- read_tsv(pipe(sprintf("gsutil cat %s", anc_file_path)))
anc_df <- anc_df %>% select(research_id, ancestry_pred)
ancestries <- unique(anc_df$ancestry_pred)

dat <- dat %>%
    left_join(anc_df, by = c("person_id" = "research_id"))
dat <- dat %>%
          mutate(
            ancestry_pred = factor(ancestry_pred)
      )

In [ ]:
#-- Load Phenotypes
pheno <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Phenotypes.csv")

#-- Join with phenotypes
both <- dat %>%
  left_join(pheno, by = "person_id") %>%
  mutate(
    age_at_first_AD = interval(date_of_birth, DateFirstDispense) %/% years(1)
  )

In [ ]:
#-- Set reference levels
both$Education_level = relevel(as.factor(both$Education_level), ref = "Less than a high school degree")
final <- both %>% 
    mutate(Income_quantitative =
          case_when(Income == ">= 200k" ~ 8,
                   Income == "150-200k" ~ 7,
                   Income == "100-150k" ~ 6,
                   Income == "75-100k" ~ 5,
                   Income == "50-75k" ~ 4,
                   Income == "35-50k" ~ 3,
                   Income == "25-35k" ~ 2,
                   Income == "10-25k" ~ 1,
                   TRUE ~ NA_integer_))


# CREATE CONSISTENT AD MODULATOR INDICATOR (based on any_followup)
final <- final %>%
  mutate(
    has_any_AD_modulator_ever = case_when(
      has_AD_cyp2d6_modulator_any_followup == 1L | 
      has_AD_cyp2c19_modulator_any_followup == 1L ~ 1L,
      TRUE ~ 0L
    )
  )

In [ ]:
ancestry_groups <- c("All", "European")
cyp_windows       <- c("any_followup", "baseline_only", "concom_only")

all_list <- list()

for (window in cyp_windows) {
  for (ancestry in ancestry_groups) {
    
    ## --- Filter ancestry ---
    if (ancestry == "European") {
      input_data <- final %>% 
        filter(ancestry_pred == "eur")
    } else {
      input_data <- final
    }
    
    ## --- Map window-specific CYP columns into generic names ---
    # phenoconverted phenotypes
    c19_adj_col <- paste0("CYP2C19_", window, "_phenoconverted")
    d6_adj_col  <- paste0("CYP2D6_",  window, "_phenoconverted")
    b6_adj_col  <- paste0("CYP2B6_",  window, "_phenoconverted")
    
    # AD modulator indicators (already exist in data)
    has_AD_d6_col  <- paste0("has_AD_cyp2d6_modulator_", window)
    has_AD_c19_col <- paste0("has_AD_cyp2c19_modulator_", window)
    
    # CYP2C19 modulators
    c19_s_inhib_col <- paste0(window, "_cyp2c19_strong_inhibitors")
    c19_m_inhib_col <- paste0(window, "_cyp2c19_moderate_inhibitors")
    c19_w_inhib_col <- paste0(window, "_cyp2c19_weak_inhibitors")
    c19_m_ind_col   <- paste0(window, "_cyp2c19_moderate_inducers")
    c19_w_ind_col   <- paste0(window, "_cyp2c19_weak_inducers")
    
    # CYP2D6 modulators (inhibitors only)
    d6_s_inhib_col  <- paste0(window, "_cyp2d6_strong_inhibitors")
    d6_m_inhib_col  <- paste0(window, "_cyp2d6_moderate_inhibitors")
    d6_w_inhib_col  <- paste0(window, "_cyp2d6_weak_inhibitors")
    
    # CYP2B6 modulators (inhibitors + inducers)
    b6_s_inhib_col  <- paste0(window, "_cyp2b6_strong_inhibitors")
    b6_m_inhib_col  <- paste0(window, "_cyp2b6_moderate_inhibitors")
    b6_w_inhib_col  <- paste0(window, "_cyp2b6_weak_inhibitors")
    b6_m_ind_col    <- paste0(window, "_cyp2b6_moderate_inducers")
    b6_w_ind_col    <- paste0(window, "_cyp2b6_weak_inducers")
      
    # CYP neg controls
    prav_col <- paste0(window, "_pravastatin_neg_cont")
    rosu_col <- paste0(window, "_rosuvastatin_neg_cont")
    
    input_data <- input_data %>%
      mutate(
        # window-specific phenoconverted phenotypes
        CYP2C19_adjusted = .data[[c19_adj_col]],
        CYP2D6_adjusted  = .data[[d6_adj_col]],
        CYP2B6_adjusted  = .data[[b6_adj_col]],
        # window-specific AD modulator indicators
        has_AD_cyp2d6_modulator  = .data[[has_AD_d6_col]],
        has_AD_cyp2c19_modulator = .data[[has_AD_c19_col]]
      )
    
    input_data <- input_data %>% 
      mutate(
        # CYP2C19 modulators
        cyp2c19_strong_inhib = .data[[c19_s_inhib_col]],
        cyp2c19_mod_inhib    = .data[[c19_m_inhib_col]],
        cyp2c19_weak_inhib   = .data[[c19_w_inhib_col]],
        cyp2c19_mod_ind      = .data[[c19_m_ind_col]],
        cyp2c19_weak_ind     = .data[[c19_w_ind_col]],
        
        # CYP2D6 modulators
        cyp2d6_strong_inhib  = .data[[d6_s_inhib_col]],
        cyp2d6_mod_inhib     = .data[[d6_m_inhib_col]],
        cyp2d6_weak_inhib    = .data[[d6_w_inhib_col]],
        
        # CYP2B6 modulators
        cyp2b6_strong_inhib  = .data[[b6_s_inhib_col]],
        cyp2b6_mod_inhib     = .data[[b6_m_inhib_col]],
        cyp2b6_weak_inhib    = .data[[b6_w_inhib_col]],
        cyp2b6_mod_ind       = .data[[b6_m_ind_col]],
        cyp2b6_weak_ind      = .data[[b6_w_ind_col]],
          
        # CYP negative controls
        prav_neg_cont        = .data[[prav_col]],
        rosu_neg_cont        = .data[[rosu_col]],
        
        # Combined AD modulator indicator
        has_any_AD_modulator = case_when(
          has_AD_cyp2d6_modulator == 1L | has_AD_cyp2c19_modulator == 1L ~ 1L,
          TRUE ~ 0L
        )
      ) 
      
      ## --- Create binary outcome ---
    input_data <- input_data %>%
      mutate(
        status = case_when(
          FinalClassification.treatment == "Switching" ~ 1L,
          FinalClassification.treatment != "Switching" ~ 0L,
          TRUE ~ NA_integer_
        )
      )
    
    ## --- Set factor levels & references ---
    input_data$CYP2C19_adjusted <- factor(input_data$CYP2C19_adjusted)
    input_data$CYP2C19_adjusted <- relevel(
      input_data$CYP2C19_adjusted,
      ref = "Normal"
    )
    input_data$CYP2D6_adjusted <- factor(input_data$CYP2D6_adjusted)
    input_data$CYP2D6_adjusted <- relevel(
      input_data$CYP2D6_adjusted,
      ref = "Normal"
    )
    input_data$CYP2B6_adjusted <- factor(input_data$CYP2B6_adjusted)
    input_data$CYP2B6_adjusted <- relevel(
      input_data$CYP2B6_adjusted,
      ref = "Normal"
    )
    
    input_data$CYP2C19 <- factor(input_data$CYP2C19)
    input_data$CYP2C19 <- relevel(
      input_data$CYP2C19,
      ref = "Normal"
    )
    input_data$CYP2D6 <- factor(input_data$CYP2D6)
    input_data$CYP2D6 <- relevel(
      input_data$CYP2D6,
      ref = "Normal"
    )
    input_data$CYP2B6 <- factor(input_data$CYP2B6)
    input_data$CYP2B6 <- relevel(
      input_data$CYP2B6,
      ref = "Normal"
    )
      
  input_data$ancestry_pred <- factor(input_data$ancestry_pred)
  input_data$ancestry_pred <- relevel(input_data$ancestry_pred, ref = "eur")
    
    
    ## --- Build base complete-case dataset for all models ---
    common_vars <- c(
      "person_id", "status",
      "age_at_first_AD", "sex_at_birth",
      "Income_quantitative", "Education_level",
      # detailed modulators
      "cyp2d6_strong_inhib", "cyp2d6_mod_inhib", "cyp2d6_weak_inhib",
      "cyp2c19_strong_inhib", "cyp2c19_mod_inhib", "cyp2c19_weak_inhib",
      "cyp2c19_mod_ind", "cyp2c19_weak_ind",
      "cyp2b6_strong_inhib", "cyp2b6_mod_inhib", "cyp2b6_weak_inhib",
       "cyp2b6_mod_ind", "cyp2b6_weak_ind",
      # cyp neg controls
      "prav_neg_cont", "rosu_neg_cont",
      # raw phenotypes
      "CYP2D6", "CYP2C19", "CYP2B6",
      # adjusted phenotypes
      "CYP2D6_adjusted", "CYP2C19_adjusted", "CYP2B6_adjusted",
      "has_AD_cyp2d6_modulator", "has_AD_cyp2c19_modulator", "has_any_AD_modulator", "has_any_AD_modulator_ever"
    )
      
    ## DIAGNOSTIC: Check which columns are killing us
    message(sprintf("\n=== DETAILED NA CHECK: Pop: %s, Window: %s ===", ancestry, window))
    message("Rows in input_data: ", nrow(input_data))

    if (ancestry == "European") {
      check_vars <- common_vars
    } else {
      check_vars <- c(common_vars, "ancestry_pred")
    }

    message("\nVariables with missing data:")
    for (var in check_vars) {
      if (var %in% names(input_data)) {
        n_na <- sum(is.na(input_data[[var]]))
        pct_na <- 100 * n_na / nrow(input_data)
        if (n_na > 0) {
          message(sprintf("  %s: %d NAs (%.1f%%)", var, n_na, pct_na))
        }
      } else {
        message(sprintf("  %s: COLUMN MISSING", var))
      }
    }


    if (ancestry == "European") {
      base_dat <- input_data %>%
        select(all_of(common_vars)) %>%
        filter(complete.cases(.))
    } else {
      base_dat <- input_data %>%
        select(all_of(c(common_vars, "ancestry_pred"))) %>%
        filter(complete.cases(.))
    }
      
    print(dim(base_dat))
    ## ============================================================
    ## Create subset excluding AD modulators for phenoconversion analyses
    ## ============================================================
    no_AD_mod_dat <- base_dat %>%
      filter(has_any_AD_modulator_ever == 0L)
      
    ad_check <- no_AD_mod_dat %>%
      summarise(
        any_AD_d6 = sum(has_AD_cyp2d6_modulator),
        any_AD_c19 = sum(has_AD_cyp2c19_modulator)
      )

    message(sprintf("AD modulator check in no_AD_mod_dat: D6=%d, C19=%d", 
                    ad_check$any_AD_d6, ad_check$any_AD_c19))
    
    ## Calculate sample sizes for no-AD-modulator subset
    no_AD_mod_switch_counts <- no_AD_mod_dat %>%
      group_by(status) %>%
      summarise(n = n(), .groups = "drop")
    
    no_AD_mod_n_cases    <- if(1 %in% no_AD_mod_switch_counts$status) no_AD_mod_switch_counts$n[no_AD_mod_switch_counts$status == 1] else 0
    no_AD_mod_n_controls <- if(0 %in% no_AD_mod_switch_counts$status) no_AD_mod_switch_counts$n[no_AD_mod_switch_counts$status == 0] else 0
    no_AD_mod_n_totals   <- no_AD_mod_n_cases + no_AD_mod_n_controls
    
    message(sprintf("No-AD-modulator subset: N_total=%d, N_cases=%d, N_controls=%d", 
                    no_AD_mod_n_totals, no_AD_mod_n_cases, no_AD_mod_n_controls))
      
    ## ============================================================
    ## Create subset with no modulator exposure for genotype-only analysis
    ## ============================================================
    no_mod_dat <- base_dat %>%
      filter(has_any_AD_modulator_ever == 0L) %>%
      filter(
        cyp2d6_strong_inhib == 0 & cyp2d6_mod_inhib == 0 & cyp2d6_weak_inhib == 0 &
        cyp2c19_strong_inhib == 0 & cyp2c19_mod_inhib == 0 & cyp2c19_weak_inhib == 0 &
        cyp2c19_mod_ind == 0 & cyp2c19_weak_ind == 0 &
        cyp2b6_strong_inhib == 0 & cyp2b6_mod_inhib == 0 & cyp2b6_weak_inhib == 0 &
        cyp2b6_mod_ind == 0 & cyp2b6_weak_ind == 0
      )

    ## Calculate sample sizes for no-modulator subset
    no_mod_switch_counts <- no_mod_dat %>%
      group_by(status) %>%
      summarise(n = n(), .groups = "drop")

    no_mod_n_cases    <- if(1 %in% no_mod_switch_counts$status) no_mod_switch_counts$n[no_mod_switch_counts$status == 1] else 0
    no_mod_n_controls <- if(0 %in% no_mod_switch_counts$status) no_mod_switch_counts$n[no_mod_switch_counts$status == 0] else 0
    no_mod_n_totals   <- no_mod_n_cases + no_mod_n_controls

    message(sprintf("No-modulator subset: N_total=%d, N_cases=%d, N_controls=%d", 
                    no_mod_n_totals, no_mod_n_cases, no_mod_n_controls))

    ## --- Overall switch prevalence ---
    switch_counts <- base_dat %>%
      group_by(status) %>%
      summarise(n = n(), .groups = "drop")
    
    n_cases    <- switch_counts$n[switch_counts$status == 1]
    n_controls <- switch_counts$n[switch_counts$status == 0]
    n_totals   <- n_cases + n_controls
    
    results_list <- list()

    # Check in your baseline_only window specifically
    table(base_dat$CYP2B6_adjusted, useNA = "always")
    
    ## Helper to fit glm, dropping 1-level factors
    fit_glm_and_tidy <- function(glm_formula, model_name, independent_label, 
                                  data = base_dat, n_total = n_totals, 
                                  n_case = n_cases, n_control = n_controls) {
      dat <- droplevels(data)

      # 1) Check outcome has both 0 and 1
      if (dplyr::n_distinct(dat$status, na.rm = TRUE) < 2) {
        message(
          "Skipping model ", model_name, 
          " (Pop: ", ancestry, ", Window: ", window, 
          "): outcome has <2 levels."
        )
        return(NULL)
      }

      # 2) Get predictor term labels actually in the formula
      term_labels <- attr(terms(glm_formula), "term.labels")

      # 3) Among those, find factor predictors with <2 levels
      bad_terms <- term_labels[
        sapply(term_labels, function(v) {
          is.factor(dat[[v]]) && dplyr::n_distinct(dat[[v]], na.rm = TRUE) < 2
        })
      ]

      if (length(bad_terms) > 0) {
        message(
          "Dropping factor(s) with <2 levels from formula: ",
          paste(bad_terms, collapse = ", "),
          " (Model: ", model_name, ", Pop: ", ancestry, ", Window: ", window, ")"
        )
        glm_formula <- stats::update(glm_formula,
                                     paste(". ~ . -", paste(bad_terms, collapse = " - ")))
      }

      # 4) Fit model
      glm_model <- stats::glm(
        glm_formula,
        data   = dat,
        family = stats::binomial(link = "logit")
      )

      sm <- summary(glm_model)$coefficients

      coef_vec <- sm[, "Estimate"]
      se_vec   <- sm[, "Std. Error"]
      z_vec    <- sm[, "z value"]
      p_vec    <- sm[, "Pr(>|z|)"]

      lower_95 <- coef_vec - 1.96 * se_vec
      upper_95 <- coef_vec + 1.96 * se_vec

      glm_results <- data.frame(
        variable   = rownames(sm),
        coef       = coef_vec,
        odds_ratio = exp(coef_vec),
        se         = se_vec,
        z          = z_vec,
        p_value    = p_vec,
        lower_95   = exp(lower_95),
        upper_95   = exp(upper_95),
        stringsAsFactors = FALSE
      )
      rownames(glm_results) <- NULL
        
        ## --- ADD: exposure counts for binary predictors ---
      binary_vars <- names(dat)[
        sapply(names(dat), function(v) {
          x <- dat[[v]]
          is.numeric(x) && all(x %in% c(0, 1), na.rm = TRUE)
        })
      ]

      exposure_counts <- setNames(
        lapply(binary_vars, function(v) {
          c(N_Exposed = sum(dat[[v]] == 1, na.rm = TRUE),
            N_Unexposed = sum(dat[[v]] == 0, na.rm = TRUE))
        }),
        binary_vars
      )

      glm_results$N_Exposed   <- NA_integer_
      glm_results$N_Unexposed <- NA_integer_

      for (v in binary_vars) {
        # coefficient row for a binary var is just named "v"
        idx <- which(glm_results$variable == v)
        if (length(idx) == 1) {
          glm_results$N_Exposed[idx]   <- exposure_counts[[v]]["N_Exposed"]
          glm_results$N_Unexposed[idx] <- exposure_counts[[v]]["N_Unexposed"]
        }
      }
      ## --- END addition ---

      glm_results$Model      <- model_name
      glm_results$Ancestry <- ancestry
      glm_results$Window     <- window
      glm_results$Sample     <- "All"
      glm_results$N_Total    <- n_total
      glm_results$N_Cases    <- n_case
      glm_results$N_Controls <- n_control

      glm_results
    }
    
    # ============================================================
    ## 1) Covariate-only model (no CYP phenotypes)
    ## ============================================================
    if (ancestry == "European") {
      formula_covonly <- status ~ age_at_first_AD + sex_at_birth +
        Income_quantitative + Education_level + 
        cyp2d6_strong_inhib + cyp2d6_mod_inhib + cyp2d6_weak_inhib +
        cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
        cyp2c19_mod_ind + cyp2c19_weak_ind +
        cyp2b6_strong_inhib + cyp2b6_mod_inhib + cyp2b6_weak_inhib +
        cyp2b6_mod_ind + cyp2b6_weak_ind + prav_neg_cont + rosu_neg_cont
    } else {
      formula_covonly <- status ~ age_at_first_AD + sex_at_birth +
        ancestry_pred + Income_quantitative + Education_level + 
        cyp2d6_strong_inhib + cyp2d6_mod_inhib + cyp2d6_weak_inhib +
        cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
        cyp2c19_mod_ind + cyp2c19_weak_ind +
        cyp2b6_strong_inhib + cyp2b6_mod_inhib + cyp2b6_weak_inhib +
        cyp2b6_mod_ind + cyp2b6_weak_ind + prav_neg_cont + rosu_neg_cont
    }
    
    results_list[["CovAll_NoCYP"]] <- fit_glm_and_tidy(
      glm_formula       = formula_covonly,
      model_name        = "CovAll_NoCYP",
      independent_label = "AllCovariates_NoCYP",
      data              = no_AD_mod_dat,      
      n_total           = no_AD_mod_n_totals,
      n_case            = no_AD_mod_n_cases,
      n_control         = no_AD_mod_n_controls
    )
      
    ## ============================================================
    ## 1b) Covariate-only model (interaction effect between CYP2D6 strong inhibs and age/sex, as well as CYP2C19 strong inhibs)
    ## ============================================================
    if (ancestry == "European") {
      formula_covonly <- status ~ age_at_first_AD*cyp2d6_strong_inhib + sex_at_birth*cyp2d6_strong_inhib +
        Income_quantitative + Education_level + 
        cyp2d6_mod_inhib + cyp2d6_weak_inhib +
        age_at_first_AD*cyp2c19_strong_inhib + sex_at_birth*cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
        cyp2c19_mod_ind + cyp2c19_weak_ind +
        cyp2b6_strong_inhib + cyp2b6_mod_inhib + cyp2b6_weak_inhib +
        cyp2b6_mod_ind + cyp2b6_weak_ind + prav_neg_cont + rosu_neg_cont
    } else {
      formula_covonly <- status ~ age_at_first_AD*cyp2d6_strong_inhib + sex_at_birth*cyp2d6_strong_inhib +
        ancestry_pred + Income_quantitative + Education_level + 
        cyp2d6_mod_inhib + cyp2d6_weak_inhib +
        sex_at_birth*cyp2c19_strong_inhib + age_at_first_AD*cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
        cyp2c19_mod_ind + cyp2c19_weak_ind +
        cyp2b6_strong_inhib + cyp2b6_mod_inhib + cyp2b6_weak_inhib +
        cyp2b6_mod_ind + cyp2b6_weak_ind + prav_neg_cont + rosu_neg_cont
    }
    
    results_list[["CovAll_NoCYP_Int"]] <- fit_glm_and_tidy(
      glm_formula       = formula_covonly,
      model_name        = "CovAll_NoCYP_Int",
      independent_label = "AllCovariates_NoCYP_Interactions",
      data              = no_AD_mod_dat, 
      n_total           = no_AD_mod_n_totals,
      n_case            = no_AD_mod_n_cases,
      n_control         = no_AD_mod_n_controls
    )
    
    
    ## ============================================================
    ## 2) CYP main-effect models (raw + adjusted, incl. CYP2B6)
    ## ============================================================
      
    ## 2d) Raw phenotypes and no modulators
    if (ancestry == "European") {
      formula_cyp_main_raw <- status ~ age_at_first_AD + sex_at_birth +
        Income_quantitative + Education_level + 
        CYP2D6 + CYP2C19 + CYP2B6
    } else {
      formula_cyp_main_raw <- status ~ age_at_first_AD + sex_at_birth +
        ancestry_pred + Income_quantitative + Education_level + 
        CYP2D6 + CYP2C19 + CYP2B6
    }
    
    results_list[["CYP_Main_raw"]] <- fit_glm_and_tidy(
      glm_formula       = formula_cyp_main_raw,
      model_name        = "CYP_Main_raw",
      independent_label = "CYP2C19_CYP2D6_CYP2B6_raw",
      data              = base_dat,   
      n_total           = n_totals,
      n_case            = n_cases,
      n_control         = n_controls
    )
    
    ## 2a) Raw phenotypes + modulators
    if (ancestry == "European") {
      formula_cyp_main_raw <- status ~ age_at_first_AD + sex_at_birth +
        Income_quantitative + Education_level + 
        cyp2d6_strong_inhib + cyp2d6_mod_inhib + cyp2d6_weak_inhib +
        cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
        cyp2c19_mod_ind + cyp2c19_weak_ind +
        cyp2b6_strong_inhib + cyp2b6_mod_inhib + cyp2b6_weak_inhib +
        cyp2b6_mod_ind + cyp2b6_weak_ind +
        CYP2D6 + CYP2C19 + CYP2B6
    } else {
      formula_cyp_main_raw <- status ~ age_at_first_AD + sex_at_birth +
        ancestry_pred + Income_quantitative + Education_level + 
        cyp2d6_strong_inhib + cyp2d6_mod_inhib + cyp2d6_weak_inhib +
        cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
        cyp2c19_mod_ind + cyp2c19_weak_ind +
        cyp2b6_strong_inhib + cyp2b6_mod_inhib + cyp2b6_weak_inhib +
        cyp2b6_mod_ind + cyp2b6_weak_ind +
        CYP2D6 + CYP2C19 + CYP2B6
    }
    
    results_list[["CYP_Main_raw_plus_Mod"]] <- fit_glm_and_tidy(
      glm_formula       = formula_cyp_main_raw,
      model_name        = "CYP_Main_raw_plus_Mod",
      independent_label = "CYP2C19_CYP2D6_CYP2B6_raw_plus_Mod",
      data              = no_AD_mod_dat,   
      n_total           = no_AD_mod_n_totals,
      n_case            = no_AD_mod_n_cases,
      n_control         = no_AD_mod_n_controls
    )
    
    ## 2b) Adjusted phenotypes only (no modulators to avoid double-counting)
    if (ancestry == "European") {
      formula_cyp_main_adj <- status ~ age_at_first_AD + sex_at_birth +
        Income_quantitative + Education_level + 
        CYP2D6_adjusted + CYP2C19_adjusted + CYP2B6_adjusted
    } else {
      formula_cyp_main_adj <- status ~ age_at_first_AD + sex_at_birth +
        ancestry_pred + Income_quantitative + Education_level + 
        CYP2D6_adjusted + CYP2C19_adjusted + CYP2B6_adjusted
    }
    
    results_list[["CYP_Main_adj"]] <- fit_glm_and_tidy(
      glm_formula       = formula_cyp_main_adj,
      model_name        = "CYP_Main_adj",
      independent_label = "CYP2C19_CYP2D6_CYP2B6_adj",
      data              = no_AD_mod_dat,   # EXCLUDE AD MODULATORS
      n_total           = no_AD_mod_n_totals,
      n_case            = no_AD_mod_n_cases,
      n_control         = no_AD_mod_n_controls
    )
      
      
    ## 2c) Adjusted phenotypes + modulators (full model)
    if (ancestry == "European") {
      formula_cyp_adj_plus_mod <- status ~ age_at_first_AD + sex_at_birth +
        Income_quantitative + Education_level + 
        cyp2d6_strong_inhib + cyp2d6_mod_inhib + cyp2d6_weak_inhib +
        cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
        cyp2c19_mod_ind + cyp2c19_weak_ind +
        cyp2b6_strong_inhib + cyp2b6_mod_inhib + cyp2b6_weak_inhib +
        cyp2b6_mod_ind + cyp2b6_weak_ind +
        CYP2D6_adjusted + CYP2C19_adjusted + CYP2B6_adjusted
    } else {
      formula_cyp_adj_plus_mod <- status ~ age_at_first_AD + sex_at_birth +
        ancestry_pred + Income_quantitative + Education_level + 
        cyp2d6_strong_inhib + cyp2d6_mod_inhib + cyp2d6_weak_inhib +
        cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
        cyp2c19_mod_ind + cyp2c19_weak_ind +
        cyp2b6_strong_inhib + cyp2b6_mod_inhib + cyp2b6_weak_inhib +
        cyp2b6_mod_ind + cyp2b6_weak_ind +
        CYP2D6_adjusted + CYP2C19_adjusted + CYP2B6_adjusted
    }
      
    results_list[["CYP_Main_adj_plus_Mod"]] <- fit_glm_and_tidy(
      glm_formula       = formula_cyp_adj_plus_mod,
      model_name        = "CYP_Main_adj_plus_Mod",
      independent_label = "CYP2C19_CYP2D6_CYP2B6_adj_plus_modulators",
      data              = no_AD_mod_dat, 
      n_total           = no_AD_mod_n_totals,
      n_case            = no_AD_mod_n_cases,
      n_control         = no_AD_mod_n_controls
    )
    
    ## ============================================================
    ## 2d) Raw genotypes in no-modulator subset (genotype-only, no modulator users)
    ## ============================================================
    if (no_mod_n_totals > 0 && no_mod_n_cases >= 20 && no_mod_n_controls >= 20) {
      if (ancestry == "European") {
        formula_cyp_no_mod <- status ~ age_at_first_AD + sex_at_birth +
          Income_quantitative + Education_level + 
          CYP2D6 + CYP2C19 + CYP2B6
      } else {
        formula_cyp_no_mod <- status ~ age_at_first_AD + sex_at_birth +
          ancestry_pred + Income_quantitative + Education_level + 
          CYP2D6 + CYP2C19 + CYP2B6
      }

      results_list[["CYP_Main_raw_NoMod"]] <- fit_glm_and_tidy(
        glm_formula       = formula_cyp_no_mod,
        model_name        = "CYP_Main_raw_NoMod",
        independent_label = "CYP2C19_CYP2D6_CYP2B6_NoModulatorUsers",
        data              = no_mod_dat,
        n_total           = no_mod_n_totals,
        n_case            = no_mod_n_cases,
        n_control         = no_mod_n_controls
      )
    } else {
      message("Skipping CYP_Genotype_NoModUsers: insufficient sample size (N_total=", 
              no_mod_n_totals, ", N_cases=", no_mod_n_cases, ", N_controls=", no_mod_n_controls, ")")
    }
    
    ## ------------------------------------------------------------
    ## Combine & store results for this ancestry × window
    ## ------------------------------------------------------------
    results_list <- results_list[!sapply(results_list, is.null)]
    
    # Only proceed if we have at least one successful model
    if (length(results_list) > 0) {
      results_df <- dplyr::bind_rows(results_list)
      
      results_format <- results_df %>%
        select(
          Ancestry, Window, Model, Sample,
          N_Total, N_Cases, N_Controls,
          variable, odds_ratio, lower_95, upper_95, p_value, N_Exposed, N_Unexposed
        ) %>%
        mutate(
          variable    = gsub(":", "_", variable),
          variable    = gsub("_adjusted", "_Ph", variable),
          variable    = gsub("Normal",      "_N", variable),
          variable    = gsub("Poor",        "_P", variable),
          variable    = gsub("Ultrarapid",  "_U", variable),
          variable    = gsub("Rapid",       "_R", variable),
          variable    = gsub("Intermediate","_I", variable)
        )
      
      combo <- paste0(ancestry, "_", window, "_CleanModels")
      message("Finished combo: ", combo)
      
      all_list[[combo]] <- results_format
    } else {
      message("No successful models for ",ancestry, "_", window)
    }
    
  } # end ancestry
}   # end window




In [ ]:
#-- Concatenate, reformat and save results
all_df <- bind_rows(all_list) %>%
    select(Ancestry, Window, Model,variable, odds_ratio, lower_95, upper_95, p_value, N_Total, N_Cases, N_Controls, N_Exposed, N_Unexposed) %>%
    mutate(Drug = "Pooled", .before = 1)

#-- Rename concom_only to post_index_only
all_df <- all_df %>%
    mutate(Window = case_when(
        Window == "concom_only" ~ "post_index_only",
        TRUE ~ Window))

all_df <- all_df %>%
  mutate(
    variable = gsub("Ph_", "", variable)
  )

In [ ]:
#-- FDR correction
all_df <- all_df %>%
  group_by(Model) %>%
  mutate(
    p_adj = case_when(
      # Cov models: correct only cyp* variables
      startsWith(Model, "Cov") & startsWith(variable, "cyp") ~
        p.adjust(p_value[startsWith(variable, "cyp")], method = "BH")[
          match(p_value, p_value[startsWith(variable, "cyp")])
        ],
      # non-Cov models: correct only CYP* variables
      !startsWith(Model, "Cov") & startsWith(variable, "CYP") ~
        p.adjust(p_value[startsWith(variable, "CYP")], method = "BH")[
          match(p_value, p_value[startsWith(variable, "CYP")])
        ],
      # everything else gets NA
      TRUE ~ NA_real_
    )
  ) %>%
  ungroup()

In [ ]:
table1 <- all_df %>% filter(Ancestry == "All" & !Model %in% c("CovAll_NoCYP", "CovAll_NoCYP_Int"))
table2 <- all_df %>% filter(Ancestry == "European" & !Model %in% c("CovAll_NoCYP", "CovAll_NoCYP_Int"))
table3 <- all_df %>% filter(Ancestry == "All" & Model %in% c("CovAll_NoCYP", "CovAll_NoCYP_Int"))
table4 <- all_df %>% filter(Ancestry == "European" & Model %in% c("CovAll_NoCYP", "CovAll_NoCYP_Int"))

In [ ]:
# Create new workbook
#wb <- createWorkbook()
wb <- loadWorkbook("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_CYP.xlsx")

# Add README sheet first
removeWorksheet(wb, "README")
addWorksheet(wb, "README")
readme_text <- data.frame(
  Model = c(
    "CovAll_NoCYP", 
    "CovAll_NoCYP_Int",
    "CYP_Main_raw", 
    "CYP_Main_raw_plus_Mod", 
    "CYP_Main_adj", 
    "CYP_Main_adj_plus_Mod", 
    "CYP_Main_raw_noMod"
  ),
  Description = c(
    "Covariates only (no CYP genotype)",
    "Covariates with interaction terms",
    "Genetic-inferred CYP metabolizer status",
    "Genetic-inferred CYP metabolizer status + CYP modulator covariates",
    "Phenoconverted-inferred CYP metabolizer status",
    "Phenoconverted-inferred CYP metabolizer status + CYP modulator covariates",
    "Genetic-inferred, no CYP modulator users"
  )
)
writeData(wb, "README", readme_text)

saveWorkbook(wb, "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_CYP.xlsx", overwrite=TRUE)

In [ ]:
wb <- loadWorkbook("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_CYP.xlsx")

removeWorksheet(wb, "Pooled_AllAncestries")
addWorksheet(wb, "Pooled_AllAncestries")
writeData(wb, "Pooled_AllAncestries", table1)
removeWorksheet(wb, "Pooled_European")
addWorksheet(wb, "Pooled_European")
writeData(wb, "Pooled_European", table2)
removeWorksheet(wb, "Pooled_Covariates_AllAncestries")
addWorksheet(wb, "Pooled_Covariates_AllAncestries")
writeData(wb, "Pooled_Covariates_AllAncestries", table3)
removeWorksheet(wb, "Pooled_Covariates_European")
addWorksheet(wb, "Pooled_Covariates_European")
writeData(wb, "Pooled_Covariates_European", table4)

saveWorkbook(wb, "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_CYP.xlsx", overwrite=TRUE)

In [ ]:
#-- First prescribed antidepressant 
first_drug_ever <- outcome %>%
  group_by(person_id) %>%
  filter(FirstDispense.treatment == min(FirstDispense.treatment)) %>%
  slice(1) %>%
  ungroup()

#-- Join with ancestries
dat <- first_drug_ever %>%
    left_join(anc_df, by = c("person_id" = "research_id"))
dat <- dat %>%
          mutate(
            ancestry_pred = factor(ancestry_pred)
      )

#-- Join with phenotypes
both <- dat %>%
  left_join(pheno, by = "person_id") %>%
  mutate(
    age_at_first_AD = interval(date_of_birth, DateFirstDispense) %/% years(1)
  )

#-- Set reference levels
both$Education_level = relevel(as.factor(both$Education_level), ref = "Less than a high school degree")
final <- both %>% 
    mutate(Income_quantitative =
          case_when(Income == ">= 200k" ~ 8,
                   Income == "150-200k" ~ 7,
                   Income == "100-150k" ~ 6,
                   Income == "75-100k" ~ 5,
                   Income == "50-75k" ~ 4,
                   Income == "35-50k" ~ 3,
                   Income == "25-35k" ~ 2,
                   Income == "10-25k" ~ 1,
                   TRUE ~ NA_integer_))


# CREATE CONSISTENT AD MODULATOR INDICATOR (based on any_followup)
final <- final %>%
  mutate(
    has_any_AD_modulator_ever = case_when(
      has_AD_cyp2d6_modulator_any_followup == 1L | 
      has_AD_cyp2c19_modulator_any_followup == 1L ~ 1L,
      TRUE ~ 0L
    )
  )

In [ ]:
ancestry_groups <- c("All", "European")
cyp_windows       <- c("any_followup", "baseline_only", "concom_only")

all_list <- list()

for (window in cyp_windows) {
  for (ancestry in ancestry_groups) {
    
    ## --- Filter ancestry ---
    if (ancestry == "European") {
      input_data <- final %>% 
        filter(ancestry_pred == "eur")
    } else {
      input_data <- final
    }
    
    ## --- Map window-specific CYP columns into generic names ---
    # phenoconverted phenotypes
    c19_adj_col <- paste0("CYP2C19_", window, "_phenoconverted")
    d6_adj_col  <- paste0("CYP2D6_",  window, "_phenoconverted")
    b6_adj_col  <- paste0("CYP2B6_",  window, "_phenoconverted")
    
    # AD modulator indicators (already exist in data)
    has_AD_d6_col  <- paste0("has_AD_cyp2d6_modulator_", window)
    has_AD_c19_col <- paste0("has_AD_cyp2c19_modulator_", window)
    
    # CYP2C19 modulators
    c19_s_inhib_col <- paste0(window, "_cyp2c19_strong_inhibitors")
    c19_m_inhib_col <- paste0(window, "_cyp2c19_moderate_inhibitors")
    c19_w_inhib_col <- paste0(window, "_cyp2c19_weak_inhibitors")
    c19_m_ind_col   <- paste0(window, "_cyp2c19_moderate_inducers")
    c19_w_ind_col   <- paste0(window, "_cyp2c19_weak_inducers")
    
    # CYP2D6 modulators (inhibitors only)
    d6_s_inhib_col  <- paste0(window, "_cyp2d6_strong_inhibitors")
    d6_m_inhib_col  <- paste0(window, "_cyp2d6_moderate_inhibitors")
    d6_w_inhib_col  <- paste0(window, "_cyp2d6_weak_inhibitors")
    
    # CYP2B6 modulators (inhibitors + inducers)
    b6_s_inhib_col  <- paste0(window, "_cyp2b6_strong_inhibitors")
    b6_m_inhib_col  <- paste0(window, "_cyp2b6_moderate_inhibitors")
    b6_w_inhib_col  <- paste0(window, "_cyp2b6_weak_inhibitors")
    b6_m_ind_col    <- paste0(window, "_cyp2b6_moderate_inducers")
    b6_w_ind_col    <- paste0(window, "_cyp2b6_weak_inducers")
      
    # CYP neg controls
    prav_col <- paste0(window, "_pravastatin_neg_cont")
    rosu_col <- paste0(window, "_rosuvastatin_neg_cont")
    
    input_data <- input_data %>%
      mutate(
        # window-specific phenoconverted phenotypes
        CYP2C19_adjusted = .data[[c19_adj_col]],
        CYP2D6_adjusted  = .data[[d6_adj_col]],
        CYP2B6_adjusted  = .data[[b6_adj_col]],
        # window-specific AD modulator indicators
        has_AD_cyp2d6_modulator  = .data[[has_AD_d6_col]],
        has_AD_cyp2c19_modulator = .data[[has_AD_c19_col]]
      )
    
    input_data <- input_data %>% 
      mutate(
        # CYP2C19 modulators
        cyp2c19_strong_inhib = .data[[c19_s_inhib_col]],
        cyp2c19_mod_inhib    = .data[[c19_m_inhib_col]],
        cyp2c19_weak_inhib   = .data[[c19_w_inhib_col]],
        cyp2c19_mod_ind      = .data[[c19_m_ind_col]],
        cyp2c19_weak_ind     = .data[[c19_w_ind_col]],
        
        # CYP2D6 modulators
        cyp2d6_strong_inhib  = .data[[d6_s_inhib_col]],
        cyp2d6_mod_inhib     = .data[[d6_m_inhib_col]],
        cyp2d6_weak_inhib    = .data[[d6_w_inhib_col]],
        
        # CYP2B6 modulators
        cyp2b6_strong_inhib  = .data[[b6_s_inhib_col]],
        cyp2b6_mod_inhib     = .data[[b6_m_inhib_col]],
        cyp2b6_weak_inhib    = .data[[b6_w_inhib_col]],
        cyp2b6_mod_ind       = .data[[b6_m_ind_col]],
        cyp2b6_weak_ind      = .data[[b6_w_ind_col]],
          
        # CYP negative controls
        prav_neg_cont        = .data[[prav_col]],
        rosu_neg_cont        = .data[[rosu_col]],
        
        # Combined AD modulator indicator
        has_any_AD_modulator = case_when(
          has_AD_cyp2d6_modulator == 1L | has_AD_cyp2c19_modulator == 1L ~ 1L,
          TRUE ~ 0L
        )
      ) 
      
      ## --- Create binary outcome ---
    input_data <- input_data %>%
      mutate(
        status = case_when(
          FinalClassification.treatment == "Switching" ~ 1L,
          FinalClassification.treatment != "Switching" ~ 0L,
          TRUE ~ NA_integer_
        )
      )
    
    ## --- Set factor levels & references ---
    input_data$CYP2C19_adjusted <- factor(input_data$CYP2C19_adjusted)
    input_data$CYP2C19_adjusted <- relevel(
      input_data$CYP2C19_adjusted,
      ref = "Normal"
    )
    input_data$CYP2D6_adjusted <- factor(input_data$CYP2D6_adjusted)
    input_data$CYP2D6_adjusted <- relevel(
      input_data$CYP2D6_adjusted,
      ref = "Normal"
    )
    input_data$CYP2B6_adjusted <- factor(input_data$CYP2B6_adjusted)
    input_data$CYP2B6_adjusted <- relevel(
      input_data$CYP2B6_adjusted,
      ref = "Normal"
    )
    
    input_data$CYP2C19 <- factor(input_data$CYP2C19)
    input_data$CYP2C19 <- relevel(
      input_data$CYP2C19,
      ref = "Normal"
    )
    input_data$CYP2D6 <- factor(input_data$CYP2D6)
    input_data$CYP2D6 <- relevel(
      input_data$CYP2D6,
      ref = "Normal"
    )
    input_data$CYP2B6 <- factor(input_data$CYP2B6)
    input_data$CYP2B6 <- relevel(
      input_data$CYP2B6,
      ref = "Normal"
    )
    
    input_data$ancestry_pred <- factor(input_data$ancestry_pred)
    input_data$ancestry_pred <- relevel(input_data$ancestry_pred, ref = "eur")
      
    ## --- Build base complete-case dataset for all models ---
    common_vars <- c(
      "person_id", "status",
      "age_at_first_AD", "sex_at_birth",
      "Income_quantitative", "Education_level",
      # detailed modulators
      "cyp2d6_strong_inhib", "cyp2d6_mod_inhib", "cyp2d6_weak_inhib",
      "cyp2c19_strong_inhib", "cyp2c19_mod_inhib", "cyp2c19_weak_inhib",
      "cyp2c19_mod_ind", "cyp2c19_weak_ind",
      "cyp2b6_strong_inhib", "cyp2b6_mod_inhib", "cyp2b6_weak_inhib",
       "cyp2b6_mod_ind", "cyp2b6_weak_ind",
      # cyp neg controls
      "prav_neg_cont", "rosu_neg_cont",
      # raw phenotypes
      "CYP2D6", "CYP2C19", "CYP2B6",
      # adjusted phenotypes
      "CYP2D6_adjusted", "CYP2C19_adjusted", "CYP2B6_adjusted",
      "has_AD_cyp2d6_modulator", "has_AD_cyp2c19_modulator", "has_any_AD_modulator", "has_any_AD_modulator_ever"
    )
      
    ## DIAGNOSTIC: Check which columns are killing us
    message(sprintf("\n=== DETAILED NA CHECK: Pop: %s, Window: %s ===", ancestry, window))
    message("Rows in input_data: ", nrow(input_data))

    if (ancestry == "European") {
      check_vars <- common_vars
    } else {
      check_vars <- c(common_vars, "ancestry_pred")
    }

    message("\nVariables with missing data:")
    for (var in check_vars) {
      if (var %in% names(input_data)) {
        n_na <- sum(is.na(input_data[[var]]))
        pct_na <- 100 * n_na / nrow(input_data)
        if (n_na > 0) {
          message(sprintf("  %s: %d NAs (%.1f%%)", var, n_na, pct_na))
        }
      } else {
        message(sprintf("  %s: COLUMN MISSING", var))
      }
    }


    if (ancestry == "European") {
      base_dat <- input_data %>%
        select(all_of(common_vars)) %>%
        filter(complete.cases(.))
    } else {
      base_dat <- input_data %>%
        select(all_of(c(common_vars, "ancestry_pred"))) %>%
        filter(complete.cases(.))
    }
      
    print(dim(base_dat))
    ## ============================================================
    ## Create subset excluding AD modulators for phenoconversion analyses
    ## ============================================================
    no_AD_mod_dat <- base_dat %>%
      filter(has_any_AD_modulator_ever == 0L)
    
    ## Calculate sample sizes for no-AD-modulator subset
    no_AD_mod_switch_counts <- no_AD_mod_dat %>%
      group_by(status) %>%
      summarise(n = n(), .groups = "drop")
    
    no_AD_mod_n_cases    <- if(1 %in% no_AD_mod_switch_counts$status) no_AD_mod_switch_counts$n[no_AD_mod_switch_counts$status == 1] else 0
    no_AD_mod_n_controls <- if(0 %in% no_AD_mod_switch_counts$status) no_AD_mod_switch_counts$n[no_AD_mod_switch_counts$status == 0] else 0
    no_AD_mod_n_totals   <- no_AD_mod_n_cases + no_AD_mod_n_controls
    
    message(sprintf("No-AD-modulator subset: N_total=%d, N_cases=%d, N_controls=%d", 
                    no_AD_mod_n_totals, no_AD_mod_n_cases, no_AD_mod_n_controls))
      
    ## ============================================================
    ## Create subset with no modulator exposure for genotype-only analysis
    ## ============================================================
    no_mod_dat <- base_dat %>%
      filter(has_any_AD_modulator_ever == 0L) %>%
      filter(
        cyp2d6_strong_inhib == 0 & cyp2d6_mod_inhib == 0 & cyp2d6_weak_inhib == 0 &
        cyp2c19_strong_inhib == 0 & cyp2c19_mod_inhib == 0 & cyp2c19_weak_inhib == 0 &
        cyp2c19_mod_ind == 0 & cyp2c19_weak_ind == 0 &
        cyp2b6_strong_inhib == 0 & cyp2b6_mod_inhib == 0 & cyp2b6_weak_inhib == 0 &
        cyp2b6_mod_ind == 0 & cyp2b6_weak_ind == 0
      )

    ## Calculate sample sizes for no-modulator subset
    no_mod_switch_counts <- no_mod_dat %>%
      group_by(status) %>%
      summarise(n = n(), .groups = "drop")

    no_mod_n_cases    <- if(1 %in% no_mod_switch_counts$status) no_mod_switch_counts$n[no_mod_switch_counts$status == 1] else 0
    no_mod_n_controls <- if(0 %in% no_mod_switch_counts$status) no_mod_switch_counts$n[no_mod_switch_counts$status == 0] else 0
    no_mod_n_totals   <- no_mod_n_cases + no_mod_n_controls

    message(sprintf("No-modulator subset: N_total=%d, N_cases=%d, N_controls=%d", 
                    no_mod_n_totals, no_mod_n_cases, no_mod_n_controls))

    ## --- Overall switch prevalence ---
    switch_counts <- base_dat %>%
      group_by(status) %>%
      summarise(n = n(), .groups = "drop")
    
    n_cases    <- switch_counts$n[switch_counts$status == 1]
    n_controls <- switch_counts$n[switch_counts$status == 0]
    n_totals   <- n_cases + n_controls
    
    results_list <- list()

    # Check in your baseline_only window specifically
    table(base_dat$CYP2B6_adjusted, useNA = "always")
    
    ## Helper to fit glm, dropping 1-level factors
    fit_glm_and_tidy <- function(glm_formula, model_name, independent_label, 
                                  data = base_dat, n_total = n_totals, 
                                  n_case = n_cases, n_control = n_controls) {
      dat <- droplevels(data)

      # 1) Check outcome has both 0 and 1
      if (dplyr::n_distinct(dat$status, na.rm = TRUE) < 2) {
        message(
          "Skipping model ", model_name, 
          " (Pop: ", ancestry, ", Window: ", window, 
          "): outcome has <2 levels."
        )
        return(NULL)
      }

      # 2) Get predictor term labels actually in the formula
      term_labels <- attr(terms(glm_formula), "term.labels")

      # 3) Among those, find factor predictors with <2 levels
      bad_terms <- term_labels[
        sapply(term_labels, function(v) {
          is.factor(dat[[v]]) && dplyr::n_distinct(dat[[v]], na.rm = TRUE) < 2
        })
      ]

      if (length(bad_terms) > 0) {
        message(
          "Dropping factor(s) with <2 levels from formula: ",
          paste(bad_terms, collapse = ", "),
          " (Model: ", model_name, ", Pop: ", ancestry, ", Window: ", window, ")"
        )
        glm_formula <- stats::update(glm_formula,
                                     paste(". ~ . -", paste(bad_terms, collapse = " - ")))
      }

      # 4) Fit model
      glm_model <- stats::glm(
        glm_formula,
        data   = dat,
        family = stats::binomial(link = "logit")
      )

      sm <- summary(glm_model)$coefficients

      coef_vec <- sm[, "Estimate"]
      se_vec   <- sm[, "Std. Error"]
      z_vec    <- sm[, "z value"]
      p_vec    <- sm[, "Pr(>|z|)"]

      lower_95 <- coef_vec - 1.96 * se_vec
      upper_95 <- coef_vec + 1.96 * se_vec

      glm_results <- data.frame(
        variable   = rownames(sm),
        coef       = coef_vec,
        odds_ratio = exp(coef_vec),
        se         = se_vec,
        z          = z_vec,
        p_value    = p_vec,
        lower_95   = exp(lower_95),
        upper_95   = exp(upper_95),
        stringsAsFactors = FALSE
      )
      rownames(glm_results) <- NULL
    
      ## --- ADD: exposure counts for binary predictors ---
      binary_vars <- names(dat)[
        sapply(names(dat), function(v) {
          x <- dat[[v]]
          is.numeric(x) && all(x %in% c(0, 1), na.rm = TRUE)
        })
      ]

      exposure_counts <- setNames(
        lapply(binary_vars, function(v) {
          c(N_Exposed = sum(dat[[v]] == 1, na.rm = TRUE),
            N_Unexposed = sum(dat[[v]] == 0, na.rm = TRUE))
        }),
        binary_vars
      )

      glm_results$N_Exposed   <- NA_integer_
      glm_results$N_Unexposed <- NA_integer_

      for (v in binary_vars) {
        # coefficient row for a binary var is just named "v"
        idx <- which(glm_results$variable == v)
        if (length(idx) == 1) {
          glm_results$N_Exposed[idx]   <- exposure_counts[[v]]["N_Exposed"]
          glm_results$N_Unexposed[idx] <- exposure_counts[[v]]["N_Unexposed"]
        }
      }
      ## --- END addition ---

      glm_results$Model      <- model_name
      glm_results$Ancestry <- ancestry
      glm_results$Window     <- window
      glm_results$Sample     <- "All"
      glm_results$N_Total    <- n_total
      glm_results$N_Cases    <- n_case
      glm_results$N_Controls <- n_control

      glm_results
    }
    
    # ============================================================
    ## 1) Covariate-only model (no CYP phenotypes)
    ## ============================================================
    if (ancestry == "European") {
      formula_covonly <- status ~ age_at_first_AD + sex_at_birth +
        Income_quantitative + Education_level + 
        cyp2d6_strong_inhib + cyp2d6_mod_inhib + cyp2d6_weak_inhib +
        cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
        cyp2c19_mod_ind + cyp2c19_weak_ind +
        cyp2b6_strong_inhib + cyp2b6_mod_inhib + cyp2b6_weak_inhib +
        cyp2b6_mod_ind + cyp2b6_weak_ind + prav_neg_cont + rosu_neg_cont
    } else {
      formula_covonly <- status ~ age_at_first_AD + sex_at_birth +
        ancestry_pred + Income_quantitative + Education_level + 
        cyp2d6_strong_inhib + cyp2d6_mod_inhib + cyp2d6_weak_inhib +
        cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
        cyp2c19_mod_ind + cyp2c19_weak_ind +
        cyp2b6_strong_inhib + cyp2b6_mod_inhib + cyp2b6_weak_inhib +
        cyp2b6_mod_ind + cyp2b6_weak_ind + prav_neg_cont + rosu_neg_cont
    }
    
    results_list[["CovAll_NoCYP"]] <- fit_glm_and_tidy(
      glm_formula       = formula_covonly,
      model_name        = "CovAll_NoCYP",
      independent_label = "AllCovariates_NoCYP",
      data              = no_AD_mod_dat,      
      n_total           = no_AD_mod_n_totals,
      n_case            = no_AD_mod_n_cases,
      n_control         = no_AD_mod_n_controls
    )
      
    ## ============================================================
    ## 1b) Covariate-only model (interaction effect between CYP2D6 strong inhibs and age/sex, as well as CYP2C19 strong inhibs)
    ## ============================================================
    if (ancestry == "European") {
      formula_covonly <- status ~ age_at_first_AD*cyp2d6_strong_inhib + sex_at_birth*cyp2d6_strong_inhib +
        Income_quantitative + Education_level + 
        cyp2d6_mod_inhib + cyp2d6_weak_inhib +
        age_at_first_AD*cyp2c19_strong_inhib + sex_at_birth*cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
        cyp2c19_mod_ind + cyp2c19_weak_ind +
        cyp2b6_strong_inhib + cyp2b6_mod_inhib + cyp2b6_weak_inhib +
        cyp2b6_mod_ind + cyp2b6_weak_ind + prav_neg_cont + rosu_neg_cont
    } else {
      formula_covonly <- status ~ age_at_first_AD*cyp2d6_strong_inhib + sex_at_birth*cyp2d6_strong_inhib +
        ancestry_pred + Income_quantitative + Education_level + 
        cyp2d6_mod_inhib + cyp2d6_weak_inhib +
        sex_at_birth*cyp2c19_strong_inhib + age_at_first_AD*cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
        cyp2c19_mod_ind + cyp2c19_weak_ind +
        cyp2b6_strong_inhib + cyp2b6_mod_inhib + cyp2b6_weak_inhib +
        cyp2b6_mod_ind + cyp2b6_weak_ind + prav_neg_cont + rosu_neg_cont
    }
    
    results_list[["CovAll_NoCYP_Int"]] <- fit_glm_and_tidy(
      glm_formula       = formula_covonly,
      model_name        = "CovAll_NoCYP_Int",
      independent_label = "AllCovariates_NoCYP_Interactions",
      data              = no_AD_mod_dat, 
      n_total           = no_AD_mod_n_totals,
      n_case            = no_AD_mod_n_cases,
      n_control         = no_AD_mod_n_controls
    )
    
    
    ## ============================================================
    ## 2) CYP main-effect models (raw + adjusted, incl. CYP2B6)
    ## ============================================================
      
    ## 2d) Raw phenotypes and no modulators
    if (ancestry == "European") {
      formula_cyp_main_raw <- status ~ age_at_first_AD + sex_at_birth +
        Income_quantitative + Education_level + 
        CYP2D6 + CYP2C19 + CYP2B6
    } else {
      formula_cyp_main_raw <- status ~ age_at_first_AD + sex_at_birth +
        ancestry_pred + Income_quantitative + Education_level + 
        CYP2D6 + CYP2C19 + CYP2B6
    }
    
    results_list[["CYP_Main_raw"]] <- fit_glm_and_tidy(
      glm_formula       = formula_cyp_main_raw,
      model_name        = "CYP_Main_raw",
      independent_label = "CYP2C19_CYP2D6_CYP2B6_raw",
      data              = base_dat,   
      n_total           = n_totals,
      n_case            = n_cases,
      n_control         = n_controls
    )
    
    ## 2a) Raw phenotypes + modulators
    if (ancestry == "European") {
      formula_cyp_main_raw <- status ~ age_at_first_AD + sex_at_birth +
        Income_quantitative + Education_level + 
        cyp2d6_strong_inhib + cyp2d6_mod_inhib + cyp2d6_weak_inhib +
        cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
        cyp2c19_mod_ind + cyp2c19_weak_ind +
        cyp2b6_strong_inhib + cyp2b6_mod_inhib + cyp2b6_weak_inhib +
        cyp2b6_mod_ind + cyp2b6_weak_ind +
        CYP2D6 + CYP2C19 + CYP2B6
    } else {
      formula_cyp_main_raw <- status ~ age_at_first_AD + sex_at_birth +
        ancestry_pred + Income_quantitative + Education_level + 
        cyp2d6_strong_inhib + cyp2d6_mod_inhib + cyp2d6_weak_inhib +
        cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
        cyp2c19_mod_ind + cyp2c19_weak_ind +
        cyp2b6_strong_inhib + cyp2b6_mod_inhib + cyp2b6_weak_inhib +
        cyp2b6_mod_ind + cyp2b6_weak_ind +
        CYP2D6 + CYP2C19 + CYP2B6
    }
    
    results_list[["CYP_Main_raw_plus_Mod"]] <- fit_glm_and_tidy(
      glm_formula       = formula_cyp_main_raw,
      model_name        = "CYP_Main_raw_plus_Mod",
      independent_label = "CYP2C19_CYP2D6_CYP2B6_raw_plus_Mod",
      data              = no_AD_mod_dat,   
      n_total           = no_AD_mod_n_totals,
      n_case            = no_AD_mod_n_cases,
      n_control         = no_AD_mod_n_controls
    )
    
    ## 2b) Adjusted phenotypes only (no modulators to avoid double-counting)
    if (ancestry == "European") {
      formula_cyp_main_adj <- status ~ age_at_first_AD + sex_at_birth +
        Income_quantitative + Education_level + 
        CYP2D6_adjusted + CYP2C19_adjusted + CYP2B6_adjusted
    } else {
      formula_cyp_main_adj <- status ~ age_at_first_AD + sex_at_birth +
        ancestry_pred + Income_quantitative + Education_level + 
        CYP2D6_adjusted + CYP2C19_adjusted + CYP2B6_adjusted
    }
    
    results_list[["CYP_Main_adj"]] <- fit_glm_and_tidy(
      glm_formula       = formula_cyp_main_adj,
      model_name        = "CYP_Main_adj",
      independent_label = "CYP2C19_CYP2D6_CYP2B6_adj",
      data              = no_AD_mod_dat,   # EXCLUDE AD MODULATORS
      n_total           = no_AD_mod_n_totals,
      n_case            = no_AD_mod_n_cases,
      n_control         = no_AD_mod_n_controls
    )
      
      
    ## 2c) Adjusted phenotypes + modulators (full model)
    if (ancestry == "European") {
      formula_cyp_adj_plus_mod <- status ~ age_at_first_AD + sex_at_birth +
        Income_quantitative + Education_level + 
        cyp2d6_strong_inhib + cyp2d6_mod_inhib + cyp2d6_weak_inhib +
        cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
        cyp2c19_mod_ind + cyp2c19_weak_ind +
        cyp2b6_strong_inhib + cyp2b6_mod_inhib + cyp2b6_weak_inhib +
        cyp2b6_mod_ind + cyp2b6_weak_ind +
        CYP2D6_adjusted + CYP2C19_adjusted + CYP2B6_adjusted
    } else {
      formula_cyp_adj_plus_mod <- status ~ age_at_first_AD + sex_at_birth +
        ancestry_pred + Income_quantitative + Education_level + 
        cyp2d6_strong_inhib + cyp2d6_mod_inhib + cyp2d6_weak_inhib +
        cyp2c19_strong_inhib + cyp2c19_mod_inhib + cyp2c19_weak_inhib +
        cyp2c19_mod_ind + cyp2c19_weak_ind +
        cyp2b6_strong_inhib + cyp2b6_mod_inhib + cyp2b6_weak_inhib +
        cyp2b6_mod_ind + cyp2b6_weak_ind +
        CYP2D6_adjusted + CYP2C19_adjusted + CYP2B6_adjusted
    }
      
    results_list[["CYP_Main_adj_plus_Mod"]] <- fit_glm_and_tidy(
      glm_formula       = formula_cyp_adj_plus_mod,
      model_name        = "CYP_Main_adj_plus_Mod",
      independent_label = "CYP2C19_CYP2D6_CYP2B6_adj_plus_modulators",
      data              = no_AD_mod_dat, 
      n_total           = no_AD_mod_n_totals,
      n_case            = no_AD_mod_n_cases,
      n_control         = no_AD_mod_n_controls
    )
    
    ## ============================================================
    ## 2d) Raw genotypes in no-modulator subset (genotype-only, no modulator users)
    ## ============================================================
    if (no_mod_n_totals > 0 && no_mod_n_cases >= 20 && no_mod_n_controls >= 20) {
      if (ancestry == "European") {
        formula_cyp_no_mod <- status ~ age_at_first_AD + sex_at_birth +
          Income_quantitative + Education_level + 
          CYP2D6 + CYP2C19 + CYP2B6
      } else {
        formula_cyp_no_mod <- status ~ age_at_first_AD + sex_at_birth +
          ancestry_pred + Income_quantitative + Education_level + 
          CYP2D6 + CYP2C19 + CYP2B6
      }

      results_list[["CYP_Main_raw_NoMod"]] <- fit_glm_and_tidy(
        glm_formula       = formula_cyp_no_mod,
        model_name        = "CYP_Main_raw_NoMod",
        independent_label = "CYP2C19_CYP2D6_CYP2B6_NoModulatorUsers",
        data              = no_mod_dat,
        n_total           = no_mod_n_totals,
        n_case            = no_mod_n_cases,
        n_control         = no_mod_n_controls
      )
    } else {
      message("Skipping CYP_Genotype_NoModUsers: insufficient sample size (N_total=", 
              no_mod_n_totals, ", N_cases=", no_mod_n_cases, ", N_controls=", no_mod_n_controls, ")")
    }
    
    ## ------------------------------------------------------------
    ## Combine & store results for this ancestry × window
    ## ------------------------------------------------------------
    results_list <- results_list[!sapply(results_list, is.null)]
    
    # Only proceed if we have at least one successful model
    if (length(results_list) > 0) {
      results_df <- dplyr::bind_rows(results_list)
      
      results_format <- results_df %>%
        select(
          Ancestry, Window, Model, Sample,
          N_Total, N_Cases, N_Controls,
          variable, odds_ratio, lower_95, upper_95, p_value, N_Exposed, N_Unexposed
        ) %>%
        mutate(
          variable    = gsub(":", "_", variable),
          variable    = gsub("_adjusted", "_Ph", variable),
          variable    = gsub("Normal",      "_N", variable),
          variable    = gsub("Poor",        "_P", variable),
          variable    = gsub("Ultrarapid",  "_U", variable),
          variable    = gsub("Rapid",       "_R", variable),
          variable    = gsub("Intermediate","_I", variable)
        )
      
      combo <- paste0(ancestry, "_", window, "_CleanModels")
      message("Finished combo: ", combo)
      
      all_list[[combo]] <- results_format
    } else {
      message("No successful models for ",ancestry, "_", window)
    }
    
  } # end ancestry
}   # end window


In [ ]:
#-- Concatenate, reformat and save results
all_df <- bind_rows(all_list) %>%
    select(Ancestry, Window, Model,variable, odds_ratio, lower_95, upper_95, p_value, N_Total, N_Cases, N_Controls, N_Exposed, N_Unexposed) %>%
    mutate(Drug = "Pooled", .before = 1)

#-- Rename concom_only to post_index_only
all_df <- all_df %>%
    mutate(Window = case_when(
        Window == "concom_only" ~ "post_index_only",
        TRUE ~ Window))

all_df <- all_df %>%
  mutate(
    variable = gsub("Ph_", "", variable)
  )

In [ ]:
#-- FDR correction
all_df <- all_df %>%
  group_by(Model) %>%
  mutate(
    p_adj = case_when(
      # Cov models: correct only cyp* variables
      startsWith(Model, "Cov") & startsWith(variable, "cyp") ~
        p.adjust(p_value[startsWith(variable, "cyp")], method = "BH")[
          match(p_value, p_value[startsWith(variable, "cyp")])
        ],
      # non-Cov models: correct only CYP* variables
      !startsWith(Model, "Cov") & startsWith(variable, "CYP") ~
        p.adjust(p_value[startsWith(variable, "CYP")], method = "BH")[
          match(p_value, p_value[startsWith(variable, "CYP")])
        ],
      # everything else gets NA
      TRUE ~ NA_real_
    )
  ) %>%
  ungroup()

In [ ]:
table1 <- all_df %>% filter(Ancestry == "European" & !Model %in% c("CovAll_NoCYP", "CovAll_NoCYP_Int"))
table2 <- all_df %>% filter(Ancestry == "All" & !Model %in% c("CovAll_NoCYP", "CovAll_NoCYP_Int"))

In [ ]:
wb <- loadWorkbook("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_CYP.xlsx")

removeWorksheet(wb, "Pooled_FirstDrug_AllAncestries")
addWorksheet(wb, "Pooled_FirstDrug_AllAncestries")
writeData(wb, "Pooled_FirstDrug_AllAncestries", table2)


saveWorkbook(wb, "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_CYP.xlsx", overwrite=TRUE)


In [ ]:
#=============== Read in results
library(readxl)
library(dplyr)
library(tidyverse)

processed_data <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_CYP.xlsx", sheet = "Pooled_European") %>%
  filter(!variable %in% c("(Intercept)"))

In [ ]:
#-- Phenoconverted (European)
main <- processed_data %>%
  filter(Model == "CYP_Main_adj" | Model == "CYP_Main_raw" | Model == "CYP_Main_raw_plus_Mod" | Model == "CYP_Main_adj_plus_Mod" | Model == "CYP_Main_raw_NoMod") %>%
  filter(str_detect(variable, "CYP")) %>%
  filter(Ancestry == "European") %>%
  mutate(Genotype = case_when(
  Model == "CYP_Main_raw" ~ "Genetic-inferred\n(Full Sample)",
  Model == "CYP_Main_raw_plus_Mod" ~ "Genetic-inferred +\nModulators Covariates", 
  Model == "CYP_Main_adj_plus_Mod" ~ "Phenoconverted-inferred +\nModulators Covariates",
  Model == "CYP_Main_raw_NoMod" ~ "Genetic-inferred\n(No Modulator Users Any Follow-up)",
  TRUE ~ "Phenoconverted-inferred")) %>%
  mutate(variable = gsub("Ph_", "", variable)) %>%
  filter(Window == "any_followup") %>%
  filter(variable != "CYP2B6_P")



# Order metabolizer status and genes
main$variable <- factor(main$variable,
   levels = c(
      "CYP2B6_P", "CYP2B6_I", "CYP2B6_N", "CYP2B6_U",
      "CYP2C19_P","CYP2C19_I","CYP2C19_R","CYP2C19_N","CYP2C19_U",
      "CYP2D6_P","CYP2D6_I","CYP2D6_N","CYP2D6_U"
))

#-- Group genes together
main <- main %>%
    mutate(Gene = case_when(
    str_detect(variable, "CYP2D6") ~ "CYP2D6",
    str_detect(variable, "CYP2B6") ~ "CYP2B6",
    TRUE ~ "CYP2C19"))

#-- Fill colour mapping
fill_colors <- c(
  "CYP2D6"  = "#c4253d",
  "CYP2C19" = "#FD8D3C",
  "CYP2B6"  = "#e9b43b"
)
main <- main %>%
  mutate(
    SigFill = if_else(p_adj < 0.05, Gene, "NotSig")
  )

#-- Facet labels with group N
main <- main %>%
    mutate(Label = paste0(Genotype, "\nN=", N_Total, " (", round((N_Cases/(N_Total))*100, 0), "%)"))
unique(main$Label)

#-- Order labels
main$Label <- factor(main$Label, levels = c("Genetic-inferred\n(Full Sample)\nN=18972 (41%)",
                                           "Genetic-inferred +\nModulators Covariates\nN=14918 (32%)",
                                           "Genetic-inferred\n(No Modulator Users Any Follow-up)\nN=3227 (24%)",
                                           "Phenoconverted-inferred\nN=14918 (32%)",
                                           "Phenoconverted-inferred +\nModulators Covariates\nN=14918 (32%)"))

In [ ]:
panelA <- main %>%
    filter(Label %in% c("Genetic-inferred\n(Full Sample)\nN=18972 (41%)", "Genetic-inferred +\nModulators Covariates\nN=14918 (32%)", 
                           "Genetic-inferred\n(No Modulator Users Any Follow-up)\nN=3227 (24%)"))
panelB <- main %>%
    filter(Label %in% c("Phenoconverted-inferred\nN=14918 (32%)", "Phenoconverted-inferred +\nModulators Covariates\nN=14918 (32%)"))

pA <- ggplot(
  panelA,
  aes(
    x = variable, y = odds_ratio,
    ymin = lower_95, ymax = upper_95
  )
) +
  coord_flip() +
  geom_hline(yintercept = 1, linetype = "dashed") +
  geom_point(
    aes(colour = Gene, fill = SigFill), 
    size = 2, 
    stroke = 1, 
    shape = 21,
    position = position_dodge(width = 0.4)
  ) +
  geom_errorbar(aes(color = Gene), width = 0.4, position = position_dodge(width = 0.4)) +
  facet_wrap(Label ~ ., ncol = 3) +
  scale_alpha_manual(values = c(`TRUE` = 1, `FALSE` = 0.4), guide = "none") +
  labs(
    x = NULL,
    y = "Odds ratio (95% CI)",
    title = NULL
  ) +
  theme_minimal() +
  theme(
    axis.text = element_text(color = "black"),
    strip.background = element_rect(fill = "grey92", colour = NA),
    legend.position = "bottom",
    axis.title = element_text(face = "bold"),
    strip.text = element_text(face = "bold"),
    legend.title = element_text(face = "bold"),
    plot.title = element_text(face = "bold")
  ) +
  scale_color_manual(values = fill_colors, name = "Gene") +
  scale_fill_manual(
    values = c(fill_colors, NotSig = "white"),
    guide = "none"
  )


pB <- ggplot(
  panelB,
  aes(
    x = variable, y = odds_ratio,
    ymin = lower_95, ymax = upper_95
  )
) +
  coord_flip() +
  geom_hline(yintercept = 1, linetype = "dashed") +
  geom_point(
    aes(colour = Gene, fill = SigFill), 
    size = 2, 
    stroke = 1, 
    shape = 21,
    position = position_dodge(width = 0.4)
  ) +
  geom_errorbar(aes(color = Gene), width = 0.4, position = position_dodge(width = 0.4)) +
  facet_wrap(Label ~ ., ncol = 2) +
  scale_alpha_manual(values = c(`TRUE` = 1, `FALSE` = 0.4), guide = "none") +
  labs(
    x = NULL,
    y = "Odds ratio (95% CI)",
    title = NULL
  ) +
  theme_minimal() +
  theme(
    axis.text = element_text(color = "black"),
    strip.background = element_rect(fill = "grey92", colour = NA),
    legend.position = "bottom",
    axis.title = element_text(face = "bold"),
    strip.text = element_text(face = "bold"),
    legend.title = element_text(face = "bold"),
    plot.title = element_text(face = "bold")
  ) +
  scale_color_manual(values = fill_colors, name = "Gene") +
  scale_fill_manual(
    values = c(fill_colors, NotSig = "white"),
    guide = "none"
  )

In [ ]:
library(cowplot)

plots <- plot_grid(pA, pB, nrow = 2, labels = c("A", "B"))
plots

In [ ]:
ggsave("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/CYP/Supp_Pooled_European.png",
       plot   = plots,
       width  = 9,
       height = 9,
       dpi = 600,
       device = "png",
       units  = "in")

In [ ]:
#--- Compare to pooled within all ancestries
processed_data2 <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_CYP.xlsx", sheet = "Pooled_AllAncestries") %>%
  filter(!variable %in% c("(Intercept)"))

# Main effects model
main <- processed_data2 %>%
  filter(Model == "CYP_Main_adj" | Model == "CYP_Main_raw" | Model == "CYP_Main_raw_plus_Mod" | Model == "CYP_Main_adj_plus_Mod" | Model == "CYP_Main_raw_NoMod") %>%
  filter(str_detect(variable, "CYP")) %>%
  filter(Ancestry == "All") %>%
  mutate(Genotype = case_when(
  Model == "CYP_Main_raw" ~ "Genetic-inferred\n(Full Sample)",
  Model == "CYP_Main_raw_plus_Mod" ~ "Genetic-inferred +\nModulators Covariates", 
  Model == "CYP_Main_adj_plus_Mod" ~ "Phenoconverted-inferred +\nModulators Covariates",
  Model == "CYP_Main_raw_NoMod" ~ "Genetic-inferred\n(No Modulator Users Any Follow-up)",
  TRUE ~ "Phenoconverted-inferred")) %>%
  mutate(variable = gsub("Ph_", "", variable)) %>%
  filter(variable != "CYP2B6_P") %>%
  filter(Window == "any_followup")


# Order metabolizer status and genes
main$variable <- factor(main$variable,
   levels = c(
      "CYP2B6_P", "CYP2B6_I", "CYP2B6_N", "CYP2B6_U",
      "CYP2C19_P","CYP2C19_I","CYP2C19_R","CYP2C19_N","CYP2C19_U",
      "CYP2D6_P","CYP2D6_I","CYP2D6_N","CYP2D6_U"
))

#-- Group genes together
main <- main %>%
    mutate(Gene = case_when(
    str_detect(variable, "CYP2D6") ~ "CYP2D6",
    str_detect(variable, "CYP2B6") ~ "CYP2B6",
    TRUE ~ "CYP2C19"))

#-- Fill colour mapping
fill_colors <- c(
  "CYP2D6"  = "#c4253d",
  "CYP2C19" = "#FD8D3C",
  "CYP2B6"  = "#e9b43b"
)
main <- main %>%
  mutate(
    SigFill = if_else(p_adj < 0.05, Gene, "NotSig")
  )

#-- Facet labels with group N
main <- main %>%
    mutate(Label = paste0(Genotype, "\nN=", N_Total, " (", round((N_Cases/(N_Total))*100, 0), "%)"))
unique(main$Label)

#-- Order labels
main$Label <- factor(main$Label, levels = c("Genetic-inferred\n(Full Sample)\nN=24837 (40%)",
                                           "Genetic-inferred +\nModulators Covariates\nN=19841 (31%)",
                                           "Genetic-inferred\n(No Modulator Users Any Follow-up)\nN=3988 (24%)",
                                           "Phenoconverted-inferred\nN=19841 (31%)",
                                           "Phenoconverted-inferred +\nModulators Covariates\nN=19841 (31%)"))

In [ ]:
panelA <- main %>%
    filter(Label %in% c("Genetic-inferred\n(Full Sample)\nN=24837 (40%)", "Genetic-inferred +\nModulators Covariates\nN=19841 (31%)", 
                           "Genetic-inferred\n(No Modulator Users Any Follow-up)\nN=3988 (24%)"))
panelB <- main %>%
    filter(Label %in% c("Phenoconverted-inferred\nN=19841 (31%)", "Phenoconverted-inferred +\nModulators Covariates\nN=19841 (31%)"))

pA <- ggplot(
  panelA,
  aes(
    x = variable, y = odds_ratio,
    ymin = lower_95, ymax = upper_95
  )
) +
  coord_flip() +
  geom_hline(yintercept = 1, linetype = "dashed") +
  geom_point(
    aes(colour = Gene, fill = SigFill), 
    size = 2, 
    stroke = 1, 
    shape = 21,
    position = position_dodge(width = 0.4)
  ) +
  geom_errorbar(aes(color = Gene), width = 0.4, position = position_dodge(width = 0.4)) +
  facet_wrap(Label ~ ., ncol = 3) +
  scale_alpha_manual(values = c(`TRUE` = 1, `FALSE` = 0.4), guide = "none") +
  labs(
    x = NULL,
    y = "Odds ratio (95% CI)",
    title = NULL
  ) +
  theme_minimal() +
  theme(
    axis.text = element_text(color = "black"),
    strip.background = element_rect(fill = "grey92", colour = NA),
    legend.position = "bottom",
    axis.title = element_text(face = "bold"),
    strip.text = element_text(face = "bold"),
    legend.title = element_text(face = "bold"),
    plot.title = element_text(face = "bold")
  ) +
  scale_color_manual(values = fill_colors, name = "Gene") +
  scale_fill_manual(
    values = c(fill_colors, NotSig = "white"),
    guide = "none"
  )


pB <- ggplot(
  panelB,
  aes(
    x = variable, y = odds_ratio,
    ymin = lower_95, ymax = upper_95
  )
) +
  coord_flip() +
  geom_hline(yintercept = 1, linetype = "dashed") +
  geom_point(
    aes(colour = Gene, fill = SigFill), 
    size = 2, 
    stroke = 1, 
    shape = 21,
    position = position_dodge(width = 0.4)
  ) +
  geom_errorbar(aes(color = Gene), width = 0.4, position = position_dodge(width = 0.4)) +
  facet_wrap(Label ~ ., ncol = 2) +
  scale_alpha_manual(values = c(`TRUE` = 1, `FALSE` = 0.4), guide = "none") +
  labs(
    x = NULL,
    y = "Odds ratio (95% CI)",
    title = NULL
  ) +
  theme_minimal() +
  theme(
    axis.text = element_text(color = "black"),
    strip.background = element_rect(fill = "grey92", colour = NA),
    legend.position = "bottom",
    axis.title = element_text(face = "bold"),
    strip.text = element_text(face = "bold"),
    legend.title = element_text(face = "bold"),
    plot.title = element_text(face = "bold")
  ) +
  scale_color_manual(values = fill_colors, name = "Gene") +
  scale_fill_manual(
    values = c(fill_colors, NotSig = "white"),
    guide = "none"
  )


In [ ]:
plots2 <- plot_grid(pA, pB, nrow = 2, labels = c("A", "B"))
plots2

In [ ]:
ggsave("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/CYP/Main_Pooled_All_Ancestries.png",
       plot   = plots2,
       width  = 9,
       height = 9,
       dpi = 600,
       device = "png",
       units  = "in")

In [ ]:
# Phenoconverted-inferred model across temporal windows (for all ancestries)
main <- processed_data2 %>%
  filter(Model == "CYP_Main_adj" | Model == "CYP_Main_raw" | Model == "CYP_Main_raw_plus_Mod" | Model == "CYP_Main_adj_plus_Mod" | Model == "CYP_Main_raw_NoMod") %>%
  filter(str_detect(variable, "CYP")) %>%
  filter(Ancestry == "All") %>%
  mutate(Genotype = case_when(
  Model == "CYP_Main_raw" ~ "Genetic-inferred\n(Full Sample)",
  Model == "CYP_Main_raw_plus_Mod" ~ "Genetic-inferred +\nModulators Covariates", 
  Model == "CYP_Main_adj_plus_Mod" ~ "Phenoconverted-inferred +\nModulators Covariates",
  Model == "CYP_Main_raw_NoMod" ~ "Genetic-inferred\n(No Modulator Users)",
  TRUE ~ "Phenoconverted-inferred")) %>%
  mutate(variable = gsub("Ph_", "", variable)) %>%
  filter(variable != "CYP2B6_P") %>%
  filter(Genotype == "Phenoconverted-inferred") %>%
  filter(Window != "any_followup")

main <- main %>%
    mutate(
    Window = case_when(
        Window == "baseline_only" ~ "Phenoconverted-inferred:\nBaseline Modulator Only",
        TRUE ~ "Phenoconverted-inferred:\nConcomitant Modulator Only"))

main$Window <- factor(main$Window, levels = c("Phenoconverted-inferred:\nBaseline Modulator Only", "Phenoconverted-inferred:\nConcomitant Modulator Only"))

#-- Group genes together
main <- main %>%
    mutate(Gene = case_when(
    str_detect(variable, "CYP2D6") ~ "CYP2D6",
    str_detect(variable, "CYP2B6") ~ "CYP2B6",
    TRUE ~ "CYP2C19"))

main <- main %>%
  mutate(
    SigFill = if_else(p_adj < 0.05, Gene, "NotSig")
  )

p2 <- ggplot(main, aes(x = variable, y = odds_ratio,
                      ymin = lower_95, ymax = upper_95,
                      colour = Gene)) +

  geom_hline(yintercept=1, linetype="dashed", size=0.7) +
  coord_flip() +
  geom_point(aes(color=Gene, fill = SigFill), shape=21, size=2, stroke=1, position=position_dodge(width=0.4)) +
  geom_errorbar(aes(color = Gene), width=0.4, position=position_dodge(width=0.4)) +
  facet_grid(.~Window) +
  theme_minimal() +
  theme(
    axis.text = element_text(color = "black"),
    strip.background = element_rect(fill="grey92", colour=NA),
    legend.position="bottom",
    axis.title = element_text(face = "bold"),
    strip.text = element_text(face = "bold"),
      legend.title = element_text(face = "bold"),
      plot.title = element_text(face = "bold")
  ) +
  labs(
    title=NULL, 
    x=NULL, y="Odds Ratio (95% CI)"
  ) +
  scale_color_manual(values = fill_colors, name = "Gene") +
  scale_fill_manual(
    values = c(fill_colors, NotSig = "white"),
    guide = "none"
  )
p2

In [ ]:
ggsave("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/CYP/Supp_Pooled_AllAncestries_TemporalWindow.png",
       plot   = p2,
       width  = 7,
       height = 5,
       dpi = 600,
       device = "png",
       units  = "in")

In [ ]:
#-- Associations with direct CYP modulator exposure (Europeans)
processed_data <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_CYP.xlsx", sheet = "Pooled_Covariates_European") %>%
  filter(!variable %in% c("(Intercept)"))

cov <- processed_data %>%
    filter(Model == "CovAll_NoCYP") %>%
    filter(Ancestry=="European")


cov <- cov %>%
    mutate(variable = 
          case_when(variable == "Education_levelTwelve Or GED" ~ "Education:Twelve/GED",
                   variable == "Education_levelCollege Undergraduate" ~ "Education:Undergraduate",
                   variable == "Education_levelCollege Postgraduate" ~ "Education:Postgraduate",
                   TRUE ~ variable))
unique(cov$variable)

cov$variable = factor(cov$variable, levels = c("cyp2b6_weak_ind", "cyp2b6_mod_ind",
                                              "cyp2b6_weak_inhib", "cyp2b6_mod_inhib", "cyp2b6_strong_inhib", "cyp2c19_weak_ind", 
                                              "cyp2c19_mod_ind", "cyp2c19_weak_inhib", "cyp2c19_mod_inhib", 
                                              "cyp2c19_strong_inhib", "cyp2d6_weak_inhib", "cyp2d6_mod_inhib", 
                                              "cyp2d6_strong_inhib","prav_neg_cont", "rosu_neg_cont",
                                               "age_at_first_AD", "sex_at_birthMale", "Income_quantitative",
                                               "Education:Twelve/GED", "Education:Undergraduate", 
                                               "Education:Postgraduate"))


# removed "cyp2b6_strong_inhib"
cov <- cov %>%
    mutate(Type = case_when(
    str_detect( variable, "cyp2d6") ~ "CYP2D6 Modulator",
    str_detect(variable, "cyp2c19") ~ "CYP2C19 Modulator",
    str_detect(variable, "cyp2b6") ~ "CYP2B6 Modulator",
    str_detect(variable, "neg") ~ "Negative Control",
    TRUE ~ "Other"))

cov <- cov %>%
    mutate(Window = case_when(
    Window == "any_followup" ~ "Modulator Window:\nAny Follow Up",
    Window == "baseline_only" ~ "Modulator Window:\nBaseline Only",
    Window == "post_index_only" ~ "Modulator Window:\nPost-Index Only",
    TRUE ~ NA_character_))

cov <- cov %>%
    filter(!(Window == "Modulator Window:\nPost-Index Only" & variable == "cyp2d6_strong_inhib"))

cov <- cov %>%
    mutate(
    Label = paste0(Window, "\nN=", N_Total, " (", round((N_Cases/N_Total)*100, 0), "%)"),
    n_label = paste0("(", N_Exposed, ")"))

cov$Label <- factor(cov$Label, levels = c("Modulator Window:\nAny Follow Up\nN=14918 (32%)", 
                                            "Modulator Window:\nBaseline Only\nN=14918 (32%)", 
                                            "Modulator Window:\nPost-Index Only\nN=14918 (32%)"))

fill_colors <- c(
  "CYP2D6 Modulator"  = "#2c7fb8",
  "CYP2C19 Modulator" = "#984ea3",
  "CYP2B6 Modulator"  = "#1a9850",
  "Negative Control"  = "#d95f02",
  "Other" = "#8b0000"
)

# add a fill variable: gene colour if Sig, "NotSig" otherwise
cov <- cov %>%
  mutate(
    SigFill = case_when(
        p_adj < 0.05 ~ Type,
        p_adj >= 0.05 ~ "NotSig",
        is.na(p_adj) ~ "NotSig",
        TRUE ~ NA_character_
  ))


In [ ]:
library(patchwork)

windows   <- unique(cov$Label)   # preserves your facet order
n_windows <- length(windows)

make_panel <- function(window_label, is_first, is_last) {
  
  df <- cov %>% filter(Label == window_label)
  
  # N_Exposed per variable (unique within a window)
  n_lookup <- df %>%
    distinct(variable, N_Exposed) %>%
    tibble::deframe()  # named vector: variable -> N_Exposed
  
  p <- ggplot(
    df,
    aes(
      x = variable, y = odds_ratio,
      ymin = lower_95, ymax = upper_95,
      colour = Type
    )
  ) +
    geom_hline(yintercept = 1, linetype = "dashed") +
    geom_point(
      aes(color = Type, fill = SigFill),
      shape = 21, size = 2, stroke = 1,
      position = position_dodge(width = 0.4)
    ) +
    geom_errorbar(
      aes(color = Type),
      width = 0.4,
      position = position_dodge(width = 0.4)
    ) +
    scale_alpha_manual(values = c(`TRUE` = 1, `FALSE` = 0.4), guide = "none") +
    scale_color_manual(values = fill_colors, name = "Gene") +
    scale_fill_manual(values = c(fill_colors, NotSig = "white"), guide = "none") +
    labs(
      x     = NULL,
      y     = if (is_last) "Odds ratio (95% CI)" else NULL,
      title = window_label
    ) +
    coord_flip() +
    theme_minimal() +
    scale_y_continuous(limits = c(0, 3), oob = scales::squish) +
    theme(
      axis.text          = element_text(color = "black"),
      axis.title         = element_text(face = "bold"),
      legend.position    = if (is_last) "bottom" else "none",
      legend.title       = element_text(face = "bold"),
      # mimic facet strip
      plot.title         = element_text(face = "bold", hjust = 0.5, size = 10),
      plot.title.position = "panel",
      panel.background   = element_rect(fill = "white",   colour = NA),
      plot.background    = element_rect(fill = "grey92",  colour = NA),
      plot.margin        = margin(t = 5, r = 5, b = 5, 
                                  l = if (is_first) 15 else 5, unit = "pt")
    )
  
  # Left panel: full variable name + N
  if (is_first) {
    p <- p + scale_x_discrete(
      labels = function(x) {
        n <- n_lookup[x]
        ifelse(is.na(n), x, paste0(x, " (N=", n, ")"))
      }
    )
  # Middle/right panels: N only, greyed out
  } else {
    p <- p +
      scale_x_discrete(
        labels = function(x) {
          n <- n_lookup[x]
          ifelse(is.na(n), "", paste0("N=", n))
        }
      ) +
      theme(axis.text.y = element_text(colour = "black", size = 8))
  }
  
  p
}

# Build all panels
panels <- lapply(seq_along(windows), function(i) {
  make_panel(windows[i], is_first = (i == 1), is_last = (i == n_windows))
})

# Combine with shared legend collected at bottom
p3 <- wrap_plots(panels, nrow = 1) +
  plot_layout(guides = "collect") &
  theme(legend.position = "bottom") &
  guides(colour = guide_legend(nrow = 2))
p3

In [ ]:
ggsave("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/CYP/Supp_Pooled_European_Modulators.png",
       plot   = p3,
       width  = 9,
       height = 7,
       dpi = 600,
       device = "png",
       units  = "in")

In [ ]:
#-- Associations with direct CYP modulator exposure (across all ancestries)
processed_data <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_CYP.xlsx", sheet = "Pooled_Covariates_AllAncestries") %>%
  filter(!variable %in% c("(Intercept)"))

cov <- processed_data %>%
    filter(Model == "CovAll_NoCYP")

cov <- cov %>%
    mutate(variable = 
          case_when(variable == "Education_levelTwelve Or GED" ~ "Education:Twelve/GED",
                   variable == "Education_levelCollege Undergraduate" ~ "Education:Undergraduate",
                   variable == "Education_levelCollege Postgraduate" ~ "Education:Postgraduate",
                   TRUE ~ variable))

cov$variable = factor(cov$variable, levels = c("cyp2b6_weak_ind", "cyp2b6_mod_ind",
                                              "cyp2b6_weak_inhib", "cyp2b6_mod_inhib", "cyp2b6_strong_inhib", "cyp2c19_weak_ind", 
                                              "cyp2c19_mod_ind", "cyp2c19_weak_inhib", "cyp2c19_mod_inhib", 
                                              "cyp2c19_strong_inhib", "cyp2d6_weak_inhib", "cyp2d6_mod_inhib", 
                                              "cyp2d6_strong_inhib","prav_neg_cont", "rosu_neg_cont",
                                               "age_at_first_AD", "sex_at_birthMale","num_prior_AD", "Income_quantitative",
                                               "Education:Twelve/GED", "Education:Undergraduate", 
                                               "Education:Postgraduate", "ancestry_predeas", "ancestry_predsas",
                                              "ancestry_predmid", "ancestry_predamr", "ancestry_predafr"))

cov <- cov %>%
    mutate(Type = case_when(
    str_detect( variable, "cyp2d6") ~ "CYP2D6 Modulator",
    str_detect(variable, "cyp2c19") ~ "CYP2C19 Modulator",
    str_detect(variable, "cyp2b6") ~ "CYP2B6 Modulator",
    str_detect(variable, "neg")   ~ "Negative Control",
    TRUE ~ "Other"))

cov <- cov %>%
    mutate(Window = case_when(
    Window == "any_followup" ~ "Modulator Window:\nAny Follow Up",
    Window == "baseline_only" ~ "Modulator Window:\nBaseline Only",
    Window == "post_index_only" ~ "Modulator Window:\nPost-Index Only",
    TRUE ~ NA_character_))

cov <- cov %>%
    filter(!(Window == "Modulator Window:\nPost-Index Only" & variable == "cyp2d6_strong_inhib"))

cov <- cov %>%
    mutate(
    Label = paste0(Window, "\nN=", N_Total, " (", round((N_Cases/N_Total)*100, 0), "%)"))

cov$Label <- factor(cov$Label, levels = c("Modulator Window:\nAny Follow Up\nN=19841 (31%)", 
                                            "Modulator Window:\nBaseline Only\nN=19841 (31%)", 
                                            "Modulator Window:\nPost-Index Only\nN=19841 (31%)"))

fill_colors <- c(
  "CYP2D6 Modulator"  = "#2c7fb8",
  "CYP2C19 Modulator" = "#984ea3",
  "CYP2B6 Modulator"  = "#1a9850",
  "Negative Control"  = "#d95f02",
  "Other" = "#8b0000"
)

# add a fill variable: gene colour if Sig, "NotSig" otherwise
cov <- cov %>%
  mutate(
    SigFill = case_when(
        p_adj < 0.05 ~ Type,
        p_adj >= 0.05 ~ "NotSig",
        is.na(p_adj) ~ "NotSig",
        TRUE ~ NA_character_
  ))

In [ ]:
library(patchwork)

windows   <- unique(cov$Label)   # preserves your facet order
n_windows <- length(windows)

make_panel <- function(window_label, is_first, is_last) {
  
  df <- cov %>% filter(Label == window_label)
  
  # N_Exposed per variable (unique within a window)
  n_lookup <- df %>%
    distinct(variable, N_Exposed) %>%
    tibble::deframe()  # named vector: variable -> N_Exposed
  
  p <- ggplot(
    df,
    aes(
      x = variable, y = odds_ratio,
      ymin = lower_95, ymax = upper_95,
      colour = Type
    )
  ) +
    geom_hline(yintercept = 1, linetype = "dashed") +
    geom_point(
      aes(color = Type, fill = SigFill),
      shape = 21, size = 2, stroke = 1,
      position = position_dodge(width = 0.4)
    ) +
    geom_errorbar(
      aes(color = Type),
      width = 0.4,
      position = position_dodge(width = 0.4)
    ) +
    scale_alpha_manual(values = c(`TRUE` = 1, `FALSE` = 0.4), guide = "none") +
    scale_color_manual(values = fill_colors, name = "Gene") +
    scale_fill_manual(values = c(fill_colors, NotSig = "white"), guide = "none") +
    labs(
      x     = NULL,
      y     = if (is_last) "Odds ratio (95% CI)" else NULL,
      title = window_label
    ) +
    coord_flip() +
    scale_y_continuous(limits = c(0, 3), oob = scales::squish) +
    theme_minimal() +
    theme(
      axis.text          = element_text(color = "black"),
      axis.title         = element_text(face = "bold"),
      legend.position    = if (is_last) "bottom" else "none",
      legend.title       = element_text(face = "bold"),
      # mimic facet strip
      plot.title         = element_text(face = "bold", hjust = 0.5, size = 10),
      plot.title.position = "panel",
      panel.background   = element_rect(fill = "white",   colour = NA),
      plot.background    = element_rect(fill = "grey92",  colour = NA),
      plot.margin        = margin(t = 5, r = 5, b = 5, 
                                  l = if (is_first) 15 else 5, unit = "pt")
    )
  
  # Left panel: full variable name + N
  if (is_first) {
    p <- p + scale_x_discrete(
      labels = function(x) {
        n <- n_lookup[x]
        ifelse(is.na(n), x, paste0(x, " (N=", n, ")"))
      }
    )
  # Middle/right panels: N only, greyed out
  } else {
    p <- p +
      scale_x_discrete(
        labels = function(x) {
          n <- n_lookup[x]
          ifelse(is.na(n), "", paste0("N=", n))
        }
      ) +
      theme(axis.text.y = element_text(colour = "black", size = 8))
  }
  
  p
}

# Build all panels
panels <- lapply(seq_along(windows), function(i) {
  make_panel(windows[i], is_first = (i == 1), is_last = (i == n_windows))
})

# Combine with shared legend collected at bottom
p3 <- wrap_plots(panels, nrow = 1) +
  plot_layout(guides = "collect") &
  theme(legend.position = "bottom") &
  guides(colour = guide_legend(nrow = 2))
p3

In [ ]:
ggsave("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/CYP/Main_Pooled_All_Ancestries_Modulators.png",
       plot   = p3,
       width  = 9,
       height = 7,
       dpi = 600,
       device = "png",
       units  = "in")

In [ ]:
#=============== Read in results
library(readxl)
library(dplyr)
library(tidyverse)

processed_data <- read_excel("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_CYP.xlsx", sheet = "Pooled_FirstDrug_AllAncestries") %>%
  filter(!variable %in% c("(Intercept)"))

In [ ]:
# Main effects model
main <- processed_data %>%
  filter(Model == "CYP_Main_adj" | Model == "CYP_Main_raw" | Model == "CYP_Main_raw_plus_Mod" | Model == "CYP_Main_adj_plus_Mod" | Model == "CYP_Main_raw_NoMod") %>%
  filter(str_detect(variable, "CYP")) %>%
  filter(Ancestry == "All") %>%
  mutate(Genotype = case_when(
  Model == "CYP_Main_raw" ~ "Genetic-inferred\n(Full Sample)",
  Model == "CYP_Main_raw_plus_Mod" ~ "Genetic-inferred +\nModulators Covariates", 
  Model == "CYP_Main_adj_plus_Mod" ~ "Phenoconverted-inferred +\nModulators Covariates",
  Model == "CYP_Main_raw_NoMod" ~ "Genetic-inferred\n(No Modulator Users)",
  TRUE ~ "Phenoconverted-inferred")) %>%
  mutate(variable = gsub("Ph_", "", variable)) %>%
  filter(variable != "CYP2B6_P") %>%
  filter(Window == "any_followup")


# Order metabolizer status and genes
main$variable <- factor(main$variable,
   levels = c(
      "CYP2B6_P", "CYP2B6_I", "CYP2B6_N", "CYP2B6_U",
      "CYP2C19_P","CYP2C19_I","CYP2C19_R","CYP2C19_N","CYP2C19_U",
      "CYP2D6_P","CYP2D6_I","CYP2D6_N","CYP2D6_U"
))

#-- Group genes together
main <- main %>%
    mutate(Gene = case_when(
    str_detect(variable, "CYP2D6") ~ "CYP2D6",
    str_detect(variable, "CYP2B6") ~ "CYP2B6",
    TRUE ~ "CYP2C19"))

#-- Fill colour mapping
fill_colors <- c(
  "CYP2D6"  = "#c4253d",
  "CYP2C19" = "#FD8D3C",
  "CYP2B6"  = "#e9b43b"
)
main <- main %>%
  mutate(
    SigFill = if_else(p_adj < 0.05, Gene, "NotSig")
  )

#-- Facet labels with group N
main <- main %>%
    mutate(Label = paste0(Genotype, "\nN=", N_Total, " (", round((N_Cases/(N_Total))*100, 0), "%)"))
unique(main$Label)

#-- Order labels
main$Label <- factor(main$Label, levels = c("Genetic-inferred\n(Full Sample)\nN=24837 (30%)",
                                           "Genetic-inferred +\nModulators Covariates\nN=19841 (23%)",
                                           "Genetic-inferred\n(No Modulator Users)\nN=3988 (19%)",
                                           "Phenoconverted-inferred\nN=19841 (23%)",
                                           "Phenoconverted-inferred +\nModulators Covariates\nN=19841 (23%)"))

In [ ]:
panelA <- main %>%
    filter(Label %in% c("Genetic-inferred\n(Full Sample)\nN=24837 (30%)", "Genetic-inferred +\nModulators Covariates\nN=19841 (23%)", 
                           "Genetic-inferred\n(No Modulator Users)\nN=3988 (19%)"))
panelB <- main %>%
    filter(Label %in% c("Phenoconverted-inferred\nN=19841 (23%)", "Phenoconverted-inferred +\nModulators Covariates\nN=19841 (23%)"))

pA <- ggplot(
  panelA,
  aes(
    x = variable, y = odds_ratio,
    ymin = lower_95, ymax = upper_95
  )
) +
  coord_flip() +
  geom_hline(yintercept = 1, linetype = "dashed") +
  geom_point(
    aes(colour = Gene, fill = SigFill), 
    size = 2, 
    stroke = 1, 
    shape = 21,
    position = position_dodge(width = 0.4)
  ) +
  geom_errorbar(aes(color = Gene), width = 0.4, position = position_dodge(width = 0.4)) +
  facet_wrap(Label ~ ., ncol = 3) +
  scale_alpha_manual(values = c(`TRUE` = 1, `FALSE` = 0.4), guide = "none") +
  labs(
    x = NULL,
    y = "Odds ratio (95% CI)",
    title = NULL
  ) +
  theme_minimal() +
  theme(
    axis.text = element_text(color = "black"),
    strip.background = element_rect(fill = "grey92", colour = NA),
    legend.position = "bottom",
    axis.title = element_text(face = "bold"),
    strip.text = element_text(face = "bold"),
    legend.title = element_text(face = "bold"),
    plot.title = element_text(face = "bold")
  ) +
  scale_color_manual(values = fill_colors, name = "Gene") +
  scale_fill_manual(
    values = c(fill_colors, NotSig = "white"),
    guide = "none"
  )


pB <- ggplot(
  panelB,
  aes(
    x = variable, y = odds_ratio,
    ymin = lower_95, ymax = upper_95
  )
) +
  coord_flip() +
  geom_hline(yintercept = 1, linetype = "dashed") +
  geom_point(
    aes(colour = Gene, fill = SigFill), 
    size = 2, 
    stroke = 1, 
    shape = 21,
    position = position_dodge(width = 0.4)
  ) +
  geom_errorbar(aes(color = Gene), width = 0.4, position = position_dodge(width = 0.4)) +
  facet_wrap(Label ~ ., ncol = 2) +
  scale_alpha_manual(values = c(`TRUE` = 1, `FALSE` = 0.4), guide = "none") +
  labs(
    x = NULL,
    y = "Odds ratio (95% CI)",
    title = NULL
  ) +
  theme_minimal() +
  theme(
    axis.text = element_text(color = "black"),
    strip.background = element_rect(fill = "grey92", colour = NA),
    legend.position = "bottom",
    axis.title = element_text(face = "bold"),
    strip.text = element_text(face = "bold"),
    legend.title = element_text(face = "bold"),
    plot.title = element_text(face = "bold")
  ) +
  scale_color_manual(values = fill_colors, name = "Gene") +
  scale_fill_manual(
    values = c(fill_colors, NotSig = "white"),
    guide = "none"
  )


In [ ]:
library(cowplot)

plots <- plot_grid(pA, pB, nrow = 2, labels = c("A", "B"))
plots

In [ ]:
ggsave("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Figures/CYP/Supp_Pooled_FirstDrug.png",
       plot   = plots,
       width  = 9,
       height = 9,
       dpi = 600,
       device = "png",
       units  = "in")

In [ ]:
# CYP non-AD Inhibitor/Inducer prevalence (excluding participants with AD modulators)
dat <- dat %>%
  mutate(
    has_any_AD_modulator_ever = case_when(
      has_AD_cyp2d6_modulator_any_followup == 1L | 
      has_AD_cyp2c19_modulator_any_followup == 1L ~ 1L,
      TRUE ~ 0L
    )
  )

mean(dat$has_any_AD_modulator_ever)
drug_data_unique <- dat %>%
    select(person_id, 
           any_followup_cyp2d6_weak_inhibitors = has_non_AD_cyp2d6_weak_inhibitors_any_followup,
           any_followup_cyp2d6_moderate_inhibitors = has_non_AD_cyp2d6_moderate_inhibitors_any_followup,
           any_followup_cyp2d6_strong_inhibitors = has_non_AD_cyp2d6_strong_inhibitors_any_followup,
           baseline_only_cyp2d6_weak_inhibitors = has_non_AD_cyp2d6_weak_inhibitors_baseline_only,
           baseline_only_cyp2d6_moderate_inhibitors = has_non_AD_cyp2d6_moderate_inhibitors_baseline_only,
           baseline_only_cyp2d6_strong_inhibitors = has_non_AD_cyp2d6_strong_inhibitors_baseline_only,
           concom_only_cyp2d6_weak_inhibitors = has_non_AD_cyp2d6_weak_inhibitors_concom_only,
           concom_only_cyp2d6_moderate_inhibitors = has_non_AD_cyp2d6_moderate_inhibitors_concom_only,
           concom_only_cyp2d6_strong_inhibitors = has_non_AD_cyp2d6_strong_inhibitors_concom_only,
           any_followup_cyp2c19_strong_inhibitors = has_non_AD_cyp2c19_strong_inhibitors_any_followup,
           any_followup_cyp2c19_moderate_inhibitors = has_non_AD_cyp2c19_moderate_inhibitors_any_followup,
           baseline_only_cyp2c19_strong_inhibitors = has_non_AD_cyp2c19_strong_inhibitors_baseline_only,
           baseline_only_cyp2c19_moderate_inhibitors = has_non_AD_cyp2c19_moderate_inhibitors_baseline_only,
           concom_only_cyp2c19_strong_inhibitors = has_non_AD_cyp2c19_strong_inhibitors_concom_only,
           concom_only_cyp2c19_moderate_inhibitors = has_non_AD_cyp2c19_moderate_inhibitors_concom_only,
           any_followup_cyp2c19_weak_inhibitors,
           any_followup_cyp2c19_moderate_inducers,
           any_followup_cyp2c19_weak_inducers,
           baseline_only_cyp2c19_weak_inhibitors,
           baseline_only_cyp2c19_moderate_inducers,
           baseline_only_cyp2c19_weak_inducers,
           concom_only_cyp2c19_weak_inhibitors,
           concom_only_cyp2c19_moderate_inducers,
           concom_only_cyp2c19_weak_inducers,
           any_followup_cyp2b6_strong_inhibitors,
           any_followup_cyp2b6_moderate_inhibitors,
           any_followup_cyp2b6_weak_inhibitors,
           any_followup_cyp2b6_moderate_inducers,
           any_followup_cyp2b6_weak_inducers,
           baseline_only_cyp2b6_strong_inhibitors,
           baseline_only_cyp2b6_moderate_inhibitors,
           baseline_only_cyp2b6_weak_inhibitors,
           baseline_only_cyp2b6_moderate_inducers,
           baseline_only_cyp2b6_weak_inducers,
           concom_only_cyp2b6_strong_inhibitors,
           concom_only_cyp2b6_moderate_inhibitors,
           concom_only_cyp2b6_weak_inhibitors,
           concom_only_cyp2b6_moderate_inducers,
           concom_only_cyp2b6_weak_inducers)


# Define parameters
cyp_genes <- c("cyp2d6", "cyp2c19", "cyp2b6")
cyp_windows <- c("any_followup", "baseline_only", "concom_only")

# Define which modulators each gene has
gene_modulators <- list(
  cyp2d6 = c("strong_inhibitors", "moderate_inhibitors", "weak_inhibitors"),
  cyp2c19 = c("strong_inhibitors", "moderate_inhibitors", "weak_inhibitors", 
              "moderate_inducers", "weak_inducers"),
  cyp2b6 = c("strong_inhibitors", "moderate_inhibitors", "weak_inhibitors",
             "moderate_inducers", "weak_inducers")
)

exposure_summary_list <- list()

#== exclude those with an AD dispensed cocomitantly with switching outcome from prevalence estimates
drug_data_unique <- dat %>%
  filter(has_any_AD_modulator_ever == 0L)

n_total <- nrow(drug_data_unique)

for (gene in cyp_genes) {
  for (window in cyp_windows) {
    
    modulators <- gene_modulators[[gene]]
    
    summary_row <- data.frame(
      Gene = toupper(gene),
      Window = window,
      N = n_total,
      stringsAsFactors = FALSE
    )
    
    any_inhib_cols <- c()
    any_ind_cols <- c()
    
    for (mod in modulators) {
      col_name <- paste0(window, "_", gene, "_", mod)
      
      n_exposed <- sum(drug_data_unique[[col_name]] == 1, na.rm = TRUE)
      pct_exposed <- round(100 * mean(drug_data_unique[[col_name]] == 1, na.rm = TRUE), 1)
      
      mod_clean <- gsub("_", " ", mod)
      mod_clean <- tools::toTitleCase(mod_clean)
      mod_clean <- gsub(" ", "_", mod_clean)
      
      summary_row[[paste0("N_", mod_clean)]] <- n_exposed
      summary_row[[paste0("Pct_", mod_clean)]] <- pct_exposed
      
      if (grepl("inhibitor", mod)) {
        any_inhib_cols <- c(any_inhib_cols, col_name)
      } else if (grepl("inducer", mod)) {
        any_ind_cols <- c(any_ind_cols, col_name)
      }
    }
    
    # Calculate "any inhibitor"
    if (length(any_inhib_cols) > 0) {
      inhib_matrix <- drug_data_unique[, any_inhib_cols, drop = FALSE] == 1
      all_na_inhib <- rowSums(is.na(drug_data_unique[, any_inhib_cols, drop = FALSE])) == length(any_inhib_cols)
      any_inhib <- rowSums(inhib_matrix, na.rm = TRUE) > 0
      any_inhib[all_na_inhib] <- NA
      summary_row$N_Any_Inhibitor <- sum(any_inhib, na.rm = TRUE)
      summary_row$Pct_Any_Inhibitor <- round(100 * mean(any_inhib, na.rm = TRUE), 1)
    }
    
    # Calculate "any inducer" (for CYP2C19 and CYP2B6)
    if (length(any_ind_cols) > 0) {
      ind_matrix <- drug_data_unique[, any_ind_cols, drop = FALSE] == 1
      all_na_ind <- rowSums(is.na(drug_data_unique[, any_ind_cols, drop = FALSE])) == length(any_ind_cols)
      any_ind <- rowSums(ind_matrix, na.rm = TRUE) > 0
      any_ind[all_na_ind] <- NA
      summary_row$N_Any_Inducer <- sum(any_ind, na.rm = TRUE)
      summary_row$Pct_Any_Inducer <- round(100 * mean(any_ind, na.rm = TRUE), 1)
    }
    
    exposure_summary_list[[paste(gene, window, sep = "_")]] <- summary_row
  }
}

# Combine all results
exposure_summary_df <- bind_rows(exposure_summary_list)
exposure_summary_df

In [ ]:
# Add "All combined" rows across all genes for each window
for (window in cyp_windows) {
  
  all_inhib_cols <- c()
  all_ind_cols <- c()
  
  for (gene in cyp_genes) {
    for (mod in gene_modulators[[gene]]) {
      col_name <- paste0(window, "_", gene, "_", mod)
      if (grepl("inhibitor", mod)) {
        all_inhib_cols <- c(all_inhib_cols, col_name)
      } else if (grepl("inducer", mod)) {
        all_ind_cols <- c(all_ind_cols, col_name)
      }
    }
  }
  
  summary_row <- data.frame(
    Gene = "ALL_COMBINED",
    Window = window,
    N = n_total,
    stringsAsFactors = FALSE
  )
  
  # Any inhibitor across all genes
  if (length(all_inhib_cols) > 0) {
    inhib_matrix <- drug_data_unique[, all_inhib_cols, drop = FALSE] == 1
    all_na_inhib <- rowSums(is.na(drug_data_unique[, all_inhib_cols, drop = FALSE])) == length(all_inhib_cols)
    any_inhib <- rowSums(inhib_matrix, na.rm = TRUE) > 0
    any_inhib[all_na_inhib] <- NA
    summary_row$N_Any_Inhibitor <- sum(any_inhib, na.rm = TRUE)
    summary_row$Pct_Any_Inhibitor <- round(100 * mean(any_inhib, na.rm = TRUE), 1)
  }
  
  # Any inducer across all genes
  if (length(all_ind_cols) > 0) {
    ind_matrix <- drug_data_unique[, all_ind_cols, drop = FALSE] == 1
    all_na_ind <- rowSums(is.na(drug_data_unique[, all_ind_cols, drop = FALSE])) == length(all_ind_cols)
    any_ind <- rowSums(ind_matrix, na.rm = TRUE) > 0
    any_ind[all_na_ind] <- NA
    summary_row$N_Any_Inducer <- sum(any_ind, na.rm = TRUE)
    summary_row$Pct_Any_Inducer <- round(100 * mean(any_ind, na.rm = TRUE), 1)
  }
  
  # Any modulator at all (inhibitor OR inducer)
  all_mod_cols <- c(all_inhib_cols, all_ind_cols)
  mod_matrix <- drug_data_unique[, all_mod_cols, drop = FALSE] == 1
  all_na_mod <- rowSums(is.na(drug_data_unique[, all_mod_cols, drop = FALSE])) == length(all_mod_cols)
  any_mod <- rowSums(mod_matrix, na.rm = TRUE) > 0
  any_mod[all_na_mod] <- NA
  summary_row$N_Any_Modulator <- sum(any_mod, na.rm = TRUE)
  summary_row$Pct_Any_Modulator <- round(100 * mean(any_mod, na.rm = TRUE), 1)
  
  exposure_summary_list[[paste("all_combined", window, sep = "_")]] <- summary_row
}

In [ ]:
# Quick check: prevalence without weak modulators
weak_cols <- grep("weak", all_mod_cols, value = TRUE)
nonweak_cols <- setdiff(all_mod_cols, weak_cols)

nonweak_matrix <- drug_data_unique[, nonweak_cols, drop = FALSE] == 1
any_nonweak <- rowSums(nonweak_matrix, na.rm = TRUE) > 0
mean(any_nonweak, na.rm = TRUE) * 100

In [ ]:
# NOW combine everything
exposure_summary_df <- bind_rows(exposure_summary_list)

exposure_summary_df <- exposure_summary_df %>%
    mutate(Window = case_when(
        Window == "concom_only" ~ "post_index_only",
        TRUE ~ Window))

wb <- loadWorkbook("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_CYP.xlsx")

removeWorksheet(wb, "Pooled_Modulator_Prevalence")
addWorksheet(wb, "Pooled_Modulator_Prevalence")
writeData(wb, "Pooled_Modulator_Prevalence", exposure_summary_df)

saveWorkbook(wb, "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_CYP.xlsx", overwrite=TRUE)


In [ ]:
exposure_summary_df

In [ ]:
# CYP non-AD Inhibitor/Inducer prevalence (without excluding participants with AD modulators)
dat <- dat %>%
  mutate(
    has_any_AD_modulator_ever = case_when(
      has_AD_cyp2d6_modulator_any_followup == 1L | 
      has_AD_cyp2c19_modulator_any_followup == 1L ~ 1L,
      TRUE ~ 0L
    )
  )


drug_data_unique <- dat %>%
    select(person_id, 
           any_followup_cyp2d6_weak_inhibitors = has_non_AD_cyp2d6_weak_inhibitors_any_followup,
           any_followup_cyp2d6_moderate_inhibitors = has_non_AD_cyp2d6_moderate_inhibitors_any_followup,
           any_followup_cyp2d6_strong_inhibitors = has_non_AD_cyp2d6_strong_inhibitors_any_followup,
           baseline_only_cyp2d6_weak_inhibitors = has_non_AD_cyp2d6_weak_inhibitors_baseline_only,
           baseline_only_cyp2d6_moderate_inhibitors = has_non_AD_cyp2d6_moderate_inhibitors_baseline_only,
           baseline_only_cyp2d6_strong_inhibitors = has_non_AD_cyp2d6_strong_inhibitors_baseline_only,
           concom_only_cyp2d6_weak_inhibitors = has_non_AD_cyp2d6_weak_inhibitors_concom_only,
           concom_only_cyp2d6_moderate_inhibitors = has_non_AD_cyp2d6_moderate_inhibitors_concom_only,
           concom_only_cyp2d6_strong_inhibitors = has_non_AD_cyp2d6_strong_inhibitors_concom_only,
           any_followup_cyp2c19_strong_inhibitors = has_non_AD_cyp2c19_strong_inhibitors_any_followup,
           any_followup_cyp2c19_moderate_inhibitors = has_non_AD_cyp2c19_moderate_inhibitors_any_followup,
           baseline_only_cyp2c19_strong_inhibitors = has_non_AD_cyp2c19_strong_inhibitors_baseline_only,
           baseline_only_cyp2c19_moderate_inhibitors = has_non_AD_cyp2c19_moderate_inhibitors_baseline_only,
           concom_only_cyp2c19_strong_inhibitors = has_non_AD_cyp2c19_strong_inhibitors_concom_only,
           concom_only_cyp2c19_moderate_inhibitors = has_non_AD_cyp2c19_moderate_inhibitors_concom_only,
           any_followup_cyp2c19_weak_inhibitors,
           any_followup_cyp2c19_moderate_inducers,
           any_followup_cyp2c19_weak_inducers,
           baseline_only_cyp2c19_weak_inhibitors,
           baseline_only_cyp2c19_moderate_inducers,
           baseline_only_cyp2c19_weak_inducers,
           concom_only_cyp2c19_weak_inhibitors,
           concom_only_cyp2c19_moderate_inducers,
           concom_only_cyp2c19_weak_inducers,
           any_followup_cyp2b6_strong_inhibitors,
           any_followup_cyp2b6_moderate_inhibitors,
           any_followup_cyp2b6_weak_inhibitors,
           any_followup_cyp2b6_moderate_inducers,
           any_followup_cyp2b6_weak_inducers,
           baseline_only_cyp2b6_strong_inhibitors,
           baseline_only_cyp2b6_moderate_inhibitors,
           baseline_only_cyp2b6_weak_inhibitors,
           baseline_only_cyp2b6_moderate_inducers,
           baseline_only_cyp2b6_weak_inducers,
           concom_only_cyp2b6_strong_inhibitors,
           concom_only_cyp2b6_moderate_inhibitors,
           concom_only_cyp2b6_weak_inhibitors,
           concom_only_cyp2b6_moderate_inducers,
           concom_only_cyp2b6_weak_inducers)


# Define parameters
cyp_genes <- c("cyp2d6", "cyp2c19", "cyp2b6")
cyp_windows <- c("any_followup", "baseline_only", "concom_only")

# Define which modulators each gene has
gene_modulators <- list(
  cyp2d6 = c("strong_inhibitors", "moderate_inhibitors", "weak_inhibitors"),
  cyp2c19 = c("strong_inhibitors", "moderate_inhibitors", "weak_inhibitors", 
              "moderate_inducers", "weak_inducers"),
  cyp2b6 = c("strong_inhibitors", "moderate_inhibitors", "weak_inhibitors",
             "moderate_inducers", "weak_inducers")
)

exposure_summary_list <- list()

#== exclude those with an AD dispensed cocomitantly with switching outcome from prevalence estimates
drug_data_unique <- dat #%>%
  #filter(has_any_AD_modulator_ever == 0L)

n_total <- nrow(drug_data_unique)

for (gene in cyp_genes) {
  for (window in cyp_windows) {
    
    modulators <- gene_modulators[[gene]]
    
    summary_row <- data.frame(
      Gene = toupper(gene),
      Window = window,
      N = n_total,
      stringsAsFactors = FALSE
    )
    
    any_inhib_cols <- c()
    any_ind_cols <- c()
    
    for (mod in modulators) {
      col_name <- paste0(window, "_", gene, "_", mod)
      
      n_exposed <- sum(drug_data_unique[[col_name]] == 1, na.rm = TRUE)
      pct_exposed <- round(100 * mean(drug_data_unique[[col_name]] == 1, na.rm = TRUE), 1)
      
      mod_clean <- gsub("_", " ", mod)
      mod_clean <- tools::toTitleCase(mod_clean)
      mod_clean <- gsub(" ", "_", mod_clean)
      
      summary_row[[paste0("N_", mod_clean)]] <- n_exposed
      summary_row[[paste0("Pct_", mod_clean)]] <- pct_exposed
      
      if (grepl("inhibitor", mod)) {
        any_inhib_cols <- c(any_inhib_cols, col_name)
      } else if (grepl("inducer", mod)) {
        any_ind_cols <- c(any_ind_cols, col_name)
      }
    }
    
    # Calculate "any inhibitor"
    if (length(any_inhib_cols) > 0) {
      inhib_matrix <- drug_data_unique[, any_inhib_cols, drop = FALSE] == 1
      all_na_inhib <- rowSums(is.na(drug_data_unique[, any_inhib_cols, drop = FALSE])) == length(any_inhib_cols)
      any_inhib <- rowSums(inhib_matrix, na.rm = TRUE) > 0
      any_inhib[all_na_inhib] <- NA
      summary_row$N_Any_Inhibitor <- sum(any_inhib, na.rm = TRUE)
      summary_row$Pct_Any_Inhibitor <- round(100 * mean(any_inhib, na.rm = TRUE), 1)
    }
    
    # Calculate "any inducer" (for CYP2C19 and CYP2B6)
    if (length(any_ind_cols) > 0) {
      ind_matrix <- drug_data_unique[, any_ind_cols, drop = FALSE] == 1
      all_na_ind <- rowSums(is.na(drug_data_unique[, any_ind_cols, drop = FALSE])) == length(any_ind_cols)
      any_ind <- rowSums(ind_matrix, na.rm = TRUE) > 0
      any_ind[all_na_ind] <- NA
      summary_row$N_Any_Inducer <- sum(any_ind, na.rm = TRUE)
      summary_row$Pct_Any_Inducer <- round(100 * mean(any_ind, na.rm = TRUE), 1)
    }
    
    exposure_summary_list[[paste(gene, window, sep = "_")]] <- summary_row
  }
}

# Combine all results
exposure_summary_df <- bind_rows(exposure_summary_list)
exposure_summary_df

In [ ]:
# Add "All combined" rows across all genes for each window
for (window in cyp_windows) {
  
  all_inhib_cols <- c()
  all_ind_cols <- c()
  
  for (gene in cyp_genes) {
    for (mod in gene_modulators[[gene]]) {
      col_name <- paste0(window, "_", gene, "_", mod)
      if (grepl("inhibitor", mod)) {
        all_inhib_cols <- c(all_inhib_cols, col_name)
      } else if (grepl("inducer", mod)) {
        all_ind_cols <- c(all_ind_cols, col_name)
      }
    }
  }
  
  summary_row <- data.frame(
    Gene = "ALL_COMBINED",
    Window = window,
    N = n_total,
    stringsAsFactors = FALSE
  )
  
  # Any inhibitor across all genes
  if (length(all_inhib_cols) > 0) {
    inhib_matrix <- drug_data_unique[, all_inhib_cols, drop = FALSE] == 1
    all_na_inhib <- rowSums(is.na(drug_data_unique[, all_inhib_cols, drop = FALSE])) == length(all_inhib_cols)
    any_inhib <- rowSums(inhib_matrix, na.rm = TRUE) > 0
    any_inhib[all_na_inhib] <- NA
    summary_row$N_Any_Inhibitor <- sum(any_inhib, na.rm = TRUE)
    summary_row$Pct_Any_Inhibitor <- round(100 * mean(any_inhib, na.rm = TRUE), 1)
  }
  
  # Any inducer across all genes
  if (length(all_ind_cols) > 0) {
    ind_matrix <- drug_data_unique[, all_ind_cols, drop = FALSE] == 1
    all_na_ind <- rowSums(is.na(drug_data_unique[, all_ind_cols, drop = FALSE])) == length(all_ind_cols)
    any_ind <- rowSums(ind_matrix, na.rm = TRUE) > 0
    any_ind[all_na_ind] <- NA
    summary_row$N_Any_Inducer <- sum(any_ind, na.rm = TRUE)
    summary_row$Pct_Any_Inducer <- round(100 * mean(any_ind, na.rm = TRUE), 1)
  }
  
  # Any modulator at all (inhibitor OR inducer)
  all_mod_cols <- c(all_inhib_cols, all_ind_cols)
  mod_matrix <- drug_data_unique[, all_mod_cols, drop = FALSE] == 1
  all_na_mod <- rowSums(is.na(drug_data_unique[, all_mod_cols, drop = FALSE])) == length(all_mod_cols)
  any_mod <- rowSums(mod_matrix, na.rm = TRUE) > 0
  any_mod[all_na_mod] <- NA
  summary_row$N_Any_Modulator <- sum(any_mod, na.rm = TRUE)
  summary_row$Pct_Any_Modulator <- round(100 * mean(any_mod, na.rm = TRUE), 1)
  
  exposure_summary_list[[paste("all_combined", window, sep = "_")]] <- summary_row
}

In [ ]:
# NOW combine everything
exposure_summary_df <- bind_rows(exposure_summary_list)

exposure_summary_df <- exposure_summary_df %>%
    mutate(Window = case_when(
        Window == "concom_only" ~ "post_index_only",
        TRUE ~ Window))

wb <- loadWorkbook("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_CYP.xlsx")

#removeWorksheet(wb, "Pooled_Modulator_Prev_Full")
addWorksheet(wb, "Pooled_Modulator_Prev_Full")
writeData(wb, "Pooled_Modulator_Prev_Full", exposure_summary_df)

saveWorkbook(wb, "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_CYP.xlsx", overwrite=TRUE)


In [ ]:
exposure_summary_df 

In [ ]:
# CYP AD Inhibitor/Inducer prevalence (only AD modulators)

# Define parameters
cyp_genes <- c("cyp2d6", "cyp2c19")
cyp_windows <- c("any_followup", "baseline_only", "concom_only")

# Define which modulators each gene has
gene_modulators <- list(
  cyp2d6 = c("strong_inhibitors", "moderate_inhibitors", "weak_inhibitors"),
  cyp2c19 = c("strong_inhibitors", "moderate_inhibitors")
)

exposure_summary_list <- list()

drug_data_unique <- dat
n_total <- nrow(drug_data_unique)

for (gene in cyp_genes) {
  for (window in cyp_windows) {
    
    modulators <- gene_modulators[[gene]]
    
    summary_row <- data.frame(
      Gene = toupper(gene),
      Window = window,
      N = n_total,
      stringsAsFactors = FALSE
    )
    
    any_inhib_cols <- c()
    
    for (mod in modulators) {

      col_name <- paste0("has_AD_", gene, "_", mod, "_", window)
      
      n_exposed <- sum(drug_data_unique[[col_name]] == 1, na.rm = TRUE)
      pct_exposed <- round(100 * mean(drug_data_unique[[col_name]] == 1, na.rm = TRUE), 1)
      
      mod_clean <- gsub("_", " ", mod)
      mod_clean <- tools::toTitleCase(mod_clean)
      mod_clean <- gsub(" ", "_", mod_clean)
      
      summary_row[[paste0("N_", mod_clean)]] <- n_exposed
      summary_row[[paste0("Pct_", mod_clean)]] <- pct_exposed
      
      if (grepl("inhibitor", mod)) {
        any_inhib_cols <- c(any_inhib_cols, col_name)
      }
    }
    
    # Calculate "any inhibitor"
    if (length(any_inhib_cols) > 0) {
      inhib_matrix <- drug_data_unique[, any_inhib_cols, drop = FALSE] == 1
      all_na_inhib <- rowSums(is.na(drug_data_unique[, any_inhib_cols, drop = FALSE])) == length(any_inhib_cols)
      any_inhib <- rowSums(inhib_matrix, na.rm = TRUE) > 0
      any_inhib[all_na_inhib] <- NA
      summary_row$N_Any_Inhibitor <- sum(any_inhib, na.rm = TRUE)
      summary_row$Pct_Any_Inhibitor <- round(100 * mean(any_inhib, na.rm = TRUE), 1)
    }
    
    exposure_summary_list[[paste(gene, window, sep = "_")]] <- summary_row
  }
}

# Combine all results
exposure_summary_df <- bind_rows(exposure_summary_list)
exposure_summary_df

In [ ]:
# Add "All combined" rows across all genes for each window
for (window in cyp_windows) {
  
  all_inhib_cols <- c()
  
  for (gene in cyp_genes) {
    for (mod in gene_modulators[[gene]]) {
      col_name <- paste0("has_AD_", gene, "_", mod, "_", window)
      if (grepl("inhibitor", mod)) {
        all_inhib_cols <- c(all_inhib_cols, col_name)
      }
    }
  }
  
  summary_row <- data.frame(
    Gene = "ALL_COMBINED",
    Window = window,
    N = n_total,
    stringsAsFactors = FALSE
  )
  
  # Any inhibitor across all genes
  if (length(all_inhib_cols) > 0) {
    inhib_matrix <- drug_data_unique[, all_inhib_cols, drop = FALSE] == 1
    all_na_inhib <- rowSums(is.na(drug_data_unique[, all_inhib_cols, drop = FALSE])) == length(all_inhib_cols)
    any_inhib <- rowSums(inhib_matrix, na.rm = TRUE) > 0
    any_inhib[all_na_inhib] <- NA
    summary_row$N_Any_Inhibitor <- sum(any_inhib, na.rm = TRUE)
    summary_row$Pct_Any_Inhibitor <- round(100 * mean(any_inhib, na.rm = TRUE), 1)
  }
  
  exposure_summary_list[[paste("all_combined", window, sep = "_")]] <- summary_row
}

In [ ]:
# NOW combine everything
exposure_summary_df <- bind_rows(exposure_summary_list)

exposure_summary_df <- exposure_summary_df %>%
    mutate(Window = case_when(
        Window == "concom_only" ~ "post_index_only",
        TRUE ~ Window))

wb <- loadWorkbook("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_CYP.xlsx")

removeWorksheet(wb, "Pooled_AD_Modulator_Prevalence")
addWorksheet(wb, "Pooled_AD_Modulator_Prevalence")
writeData(wb, "Pooled_AD_Modulator_Prevalence", exposure_summary_df)

saveWorkbook(wb, "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/All_Results_CYP.xlsx", overwrite=TRUE)


In [ ]:
exposure_summary_df